In [5]:
# -*- coding: utf-8 -*-
"""
Coarse screening (CAT + SOD merged) with 5-condition predictions on ALL rows.

Train (refit full) using your selected best models/params:
CAT:
  1) CAT_KmLow: GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3)
     Training set: CAT extremes only by fixed thresholds on log10(Km)
       keep: logKm <= CAT_KM_LO OR logKm >= CAT_KM_HI
       label: y=1 if logKm <= CAT_KM_LO else 0
  2) CAT_VmaxHigh: DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, class_weight="balanced")
     Training set: all CAT rows with Vmax>0
       label: y=1 if logVmax >= CAT_VMAX_THR else 0

SOD:
  3) SOD_KmLow: KNN(n_neighbors=5, weights="distance") + StandardScaler
     Training set: all SOD rows with Km>0
       label: median split on log10(Km), low_good => y=1 if logKm <= median
  4) SOD_VmaxHigh: XGBClassifier(300 trees, lr 0.05, depth 4, subsample 0.8, colsample 0.8, lambda 1.0, hist)
     Training set: all SOD rows with Vmax>0
       label: median split on log10(Vmax), high_good => y=1 if logVmax >= median
  5) SOD_IC50Low: XGBClassifier(same params)
     Training set: SOD rows with parseable IC50 and unit in {uM, ug/mL}
       label: fixed thresholds by unit:
         uM <= 0.109 -> 1
         ug/mL <= 0.7 -> 1

Then:
- Merge CAT+SOD to df_all
- For every row in df_all, predict probabilities for all 5 conditions
- Select rows where all five predicted labels are 1

Outputs:
OUT_DIR/
  - all_predictions_all5.csv        (all rows + 5 proba + 5 pred + hit_all5)
  - selected_all5.csv               (hit_all5 == 1)
  - models/
      cat_km_low_gb.joblib
      cat_vmax_high_dt.joblib
      sod_km_low_knn.joblib
      sod_vmax_high_xgb.joblib
      sod_ic50_low_xgb.joblib
  - coarse_screen_all5_config.json  (thresholds + train stats + sanity AUC)
"""

import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


# =========================
# 0) PATHS / CONSTANTS
# =========================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"   # <-- 改成你的
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"   # <-- 改成你的
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\coarse_screen_cat_sod_all5")  # <-- 改成你的
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

# columns
COL_DOI  = "data reference doi"
COL_KM   = "Km/mM"
COL_VMAX = "Vmax/μM s-1"
COL_IC50 = "IC50/μM"

# ----- CAT fixed thresholds (log10) from your CAT SHAP script -----
CAT_KM_LO     = 1.0253058652647702
CAT_KM_HI     = 1.8055008581584002
CAT_VMAX_THR  = 0.06519065318914162

# ----- SOD fixed thresholds for IC50 -----
SOD_THR_IC50_UM   = 0.109
SOD_THR_IC50_UGML = 0.7

# proba threshold to convert to class
PROBA_THR = 0.5


# =========================
# 1) FEATURE DEFINITIONS (canonical names AFTER normalization)
# =========================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate"
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = ["shape", "surface treatment", "dispersion medium", "Substrate"]


# =========================
# 2) COMMON UTILS
# =========================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize column names:
      - strip
      - collapse multiple spaces
    fixes:
      "main  metal proportion" -> "main metal proportion"
      "Substrate " -> "Substrate"
    """
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def ensure_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    """If some feature columns are missing, create them as NaN so the pipeline can run."""
    df = df.copy()
    for c in FEATURE_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Return X with FEATURE_COLS order, numeric coerced, 3rd metal filled 0, cats cleaned.
    """
    df_raw = ensure_feature_columns(df_raw)
    X = df_raw[FEATURE_COLS].copy()

    # numeric -> float
    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # 3rd metal triple missing -> 0.0
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        X[c] = X[c].fillna(0.0)

    # categorical clean
    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess_dense():
    """Dense one-hot + numeric impute (same style as your SHAP scripts)."""
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop"
    )


# =========================
# 3) LABEL HELPERS
# =========================
def make_label_median_log10(series: pd.Series, mode: str):
    """
    mode:
      - low_good:  log10(x) <= median -> 1
      - high_good: log10(x) >= median -> 1
    Return: (mask_valid, y_array_aligned_to_mask, threshold_log10)
    """
    v = pd.to_numeric(series, errors="coerce")
    mask = v.notna() & (v > 0)
    vals = v.loc[mask].astype(float).values
    logv = np.log10(vals)
    thr = float(np.median(logv))
    if mode == "low_good":
        y = (logv <= thr).astype(int)
    else:
        y = (logv >= thr).astype(int)
    return mask, y, thr


def parse_ic50_cell(x):
    """
    Parse IC50:
    - pure number -> uM
    - 'val + uM/μM/µM' -> uM
    - 'val + ug/mL/μg/mL/µg/mL' -> ug/mL
    """
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")
    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"
    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


def label_sod_ic50(ic50_value, ic50_unit):
    """Fixed thresholds by unit."""
    if pd.isna(ic50_value):
        return np.nan
    try:
        v = float(ic50_value)
    except Exception:
        return np.nan
    if v <= 0:
        return np.nan

    if ic50_unit == "uM":
        return int(v <= SOD_THR_IC50_UM)
    if ic50_unit == "ug/mL":
        return int(v <= SOD_THR_IC50_UGML)
    return np.nan


def safe_auc(y_true, p):
    try:
        if len(np.unique(y_true)) < 2:
            return None
        return float(roc_auc_score(y_true, p))
    except Exception:
        return None


# =========================
# 4) BUILD & TRAIN BEST MODELS
# =========================
def train_cat_models(df_cat: pd.DataFrame):
    """
    Returns:
      cat_km_pipe, cat_vmax_pipe, info(dict)
    """
    preprocess_km = make_preprocess_dense()
    preprocess_v  = make_preprocess_dense()

    # ---- CAT KmLow (extremes only) ----
    km = pd.to_numeric(df_cat.get(COL_KM), errors="coerce")
    m = km.notna() & (km > 0)
    df0 = df_cat.loc[m].copy()
    logkm = np.log10(km.loc[m].astype(float).values)

    keep = (logkm <= CAT_KM_LO) | (logkm >= CAT_KM_HI)
    df_train = df0.iloc[keep].copy()
    y_train = (logkm[keep] <= CAT_KM_LO).astype(int)

    cat_km_model = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    )
    cat_km_pipe = Pipeline([("preprocess", preprocess_km), ("model", cat_km_model)])
    cat_km_pipe.fit(prepare_features(df_train), y_train)

    auc_km = safe_auc(y_train, cat_km_pipe.predict_proba(prepare_features(df_train))[:, 1])

    # ---- CAT VmaxHigh (fixed threshold) ----
    vmax = pd.to_numeric(df_cat.get(COL_VMAX), errors="coerce")
    mv = vmax.notna() & (vmax > 0)
    dfv = df_cat.loc[mv].copy()
    logv = np.log10(vmax.loc[mv].astype(float).values)
    yv = (logv >= CAT_VMAX_THR).astype(int)

    cat_v_model = DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=2, class_weight="balanced", random_state=42
    )
    cat_v_pipe = Pipeline([("preprocess", preprocess_v), ("model", cat_v_model)])
    cat_v_pipe.fit(prepare_features(dfv), yv)

    auc_v = safe_auc(yv, cat_v_pipe.predict_proba(prepare_features(dfv))[:, 1])

    info = {
        "CAT_KmLow_train_extreme_n": int(len(df_train)),
        "CAT_KmLow_fixed_lo_log10": float(CAT_KM_LO),
        "CAT_KmLow_fixed_hi_log10": float(CAT_KM_HI),
        "CAT_KmLow_train_auc_sanity": auc_km,
        "CAT_VmaxHigh_train_n": int(len(dfv)),
        "CAT_VmaxHigh_fixed_thr_log10": float(CAT_VMAX_THR),
        "CAT_VmaxHigh_train_auc_sanity": auc_v,
    }
    return cat_km_pipe, cat_v_pipe, info


def train_sod_models(df_sod: pd.DataFrame):
    """
    Returns:
      sod_km_pipe, sod_vmax_pipe, sod_ic50_pipe, info(dict)
    """
    if not HAS_XGBOOST:
        raise RuntimeError("未检测到 xgboost，请先安装：pip install xgboost")

    preprocess_km = make_preprocess_dense()
    preprocess_v  = make_preprocess_dense()
    preprocess_ic = make_preprocess_dense()

    # ---- SOD KmLow (median split) ----
    m_km, y_km, thr_km = make_label_median_log10(df_sod.get(COL_KM), mode="low_good")
    df_km = df_sod.loc[m_km].copy()
    X_km = prepare_features(df_km)

    sod_km_pipe = Pipeline([
        ("preprocess", preprocess_km),
        ("scaler", StandardScaler(with_mean=True)),
        ("model", KNeighborsClassifier(n_neighbors=5, weights="distance")),
    ])
    sod_km_pipe.fit(X_km, y_km)
    auc_sod_km = safe_auc(y_km, sod_km_pipe.predict_proba(X_km)[:, 1])

    # ---- SOD VmaxHigh (median split) ----
    m_v, y_v, thr_v = make_label_median_log10(df_sod.get(COL_VMAX), mode="high_good")
    df_v = df_sod.loc[m_v].copy()
    X_v = prepare_features(df_v)

    sod_v_model = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", random_state=42, n_jobs=-1, verbosity=0
    )
    sod_vmax_pipe = Pipeline([("preprocess", preprocess_v), ("model", sod_v_model)])
    sod_vmax_pipe.fit(X_v, y_v)
    auc_sod_v = safe_auc(y_v, sod_vmax_pipe.predict_proba(X_v)[:, 1])

    # ---- SOD IC50Low (fixed thresholds by unit) ----
    df2 = df_sod.copy()
    parsed = df2.get(COL_IC50).apply(parse_ic50_cell)
    df2["IC50_value"] = parsed.apply(lambda t: t[0])
    df2["IC50_unit"]  = parsed.apply(lambda t: t[1])
    df2["y_IC50Low"]   = df2.apply(lambda r: label_sod_ic50(r["IC50_value"], r["IC50_unit"]), axis=1)

    df_ic = df2[df2["y_IC50Low"].notna()].copy()
    X_ic = prepare_features(df_ic)
    y_ic = df_ic["y_IC50Low"].astype(int).values

    sod_ic_model = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", random_state=42, n_jobs=-1, verbosity=0
    )
    sod_ic50_pipe = Pipeline([("preprocess", preprocess_ic), ("model", sod_ic_model)])
    sod_ic50_pipe.fit(X_ic, y_ic)
    auc_sod_ic = safe_auc(y_ic, sod_ic50_pipe.predict_proba(X_ic)[:, 1])

    info = {
        "SOD_KmLow_train_n": int(len(df_km)),
        "SOD_KmLow_median_thr_log10": float(thr_km),
        "SOD_KmLow_train_auc_sanity": auc_sod_km,
        "SOD_VmaxHigh_train_n": int(len(df_v)),
        "SOD_VmaxHigh_median_thr_log10": float(thr_v),
        "SOD_VmaxHigh_train_auc_sanity": auc_sod_v,
        "SOD_IC50Low_train_n": int(len(df_ic)),
        "SOD_IC50Low_fixed_thr_uM": float(SOD_THR_IC50_UM),
        "SOD_IC50Low_fixed_thr_ugmL": float(SOD_THR_IC50_UGML),
        "SOD_IC50Low_train_auc_sanity": auc_sod_ic,
    }
    return sod_km_pipe, sod_vmax_pipe, sod_ic50_pipe, info


# =========================
# 5) PREDICT ALL 5 CONDITIONS ON ALL ROWS
# =========================
def predict_all5(df_all: pd.DataFrame,
                 cat_km_pipe, cat_v_pipe,
                 sod_km_pipe, sod_v_pipe, sod_ic_pipe):
    X_all = prepare_features(df_all)

    p1 = cat_km_pipe.predict_proba(X_all)[:, 1]
    p2 = cat_v_pipe.predict_proba(X_all)[:, 1]
    p3 = sod_km_pipe.predict_proba(X_all)[:, 1]
    p4 = sod_v_pipe.predict_proba(X_all)[:, 1]
    p5 = sod_ic_pipe.predict_proba(X_all)[:, 1]

    out = df_all.copy()
    out["p_CAT_KmLow"] = p1
    out["p_CAT_VmaxHigh"] = p2
    out["p_SOD_KmLow"] = p3
    out["p_SOD_VmaxHigh"] = p4
    out["p_SOD_IC50Low"] = p5

    out["pred_CAT_KmLow"] = (p1 >= PROBA_THR).astype(int)
    out["pred_CAT_VmaxHigh"] = (p2 >= PROBA_THR).astype(int)
    out["pred_SOD_KmLow"] = (p3 >= PROBA_THR).astype(int)
    out["pred_SOD_VmaxHigh"] = (p4 >= PROBA_THR).astype(int)
    out["pred_SOD_IC50Low"] = (p5 >= PROBA_THR).astype(int)

    out["hit_all5"] = (
        (out["pred_CAT_KmLow"] == 1) &
        (out["pred_CAT_VmaxHigh"] == 1) &
        (out["pred_SOD_KmLow"] == 1) &
        (out["pred_SOD_VmaxHigh"] == 1) &
        (out["pred_SOD_IC50Low"] == 1)
    ).astype(int)

    return out


# =========================
# 6) MAIN
# =========================
def main():
    # load
    df_cat = normalize_columns(pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME))
    df_sod = normalize_columns(pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME))

    # ensure feature columns exist
    df_cat = ensure_feature_columns(df_cat)
    df_sod = ensure_feature_columns(df_sod)

    # source tag
    df_cat["source_dataset"] = "CAT"
    df_sod["source_dataset"] = "SOD"

    # merge all rows (we keep original columns, not only features)
    df_all = pd.concat([df_cat, df_sod], ignore_index=True)

    # train models (refit full)
    cat_km_pipe, cat_v_pipe, cat_info = train_cat_models(df_cat)
    sod_km_pipe, sod_v_pipe, sod_ic_pipe, sod_info = train_sod_models(df_sod)

    # save models
    joblib.dump(cat_km_pipe, OUT_DIR / "models" / "cat_km_low_gb.joblib")
    joblib.dump(cat_v_pipe,  OUT_DIR / "models" / "cat_vmax_high_dt.joblib")
    joblib.dump(sod_km_pipe, OUT_DIR / "models" / "sod_km_low_knn.joblib")
    joblib.dump(sod_v_pipe,  OUT_DIR / "models" / "sod_vmax_high_xgb.joblib")
    joblib.dump(sod_ic_pipe, OUT_DIR / "models" / "sod_ic50_low_xgb.joblib")

    # predict 5 conditions on all rows
    pred_all = predict_all5(df_all, cat_km_pipe, cat_v_pipe, sod_km_pipe, sod_v_pipe, sod_ic_pipe)

    # outputs
    pred_all.to_csv(OUT_DIR / "all_predictions_all5.csv", index=False, encoding="utf-8-sig")
    selected = pred_all[pred_all["hit_all5"] == 1].copy()
    selected.to_csv(OUT_DIR / "selected_all5.csv", index=False, encoding="utf-8-sig")

    # config
    config = {
        "PROBA_THR": PROBA_THR,
        "CAT_fixed_thresholds_log10": {
            "KmLow_lo": CAT_KM_LO,
            "KmLow_hi": CAT_KM_HI,
            "VmaxHigh_thr": CAT_VMAX_THR
        },
        "SOD_fixed_thresholds": {
            "IC50_uM": SOD_THR_IC50_UM,
            "IC50_ugmL": SOD_THR_IC50_UGML
        },
        "train_sanity_auc": {**cat_info, **sod_info},
        "notes": "All five models are trained on their own dataset (CAT or SOD) then applied to ALL rows in merged data."
    }
    with open(OUT_DIR / "coarse_screen_all5_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    # summary print
    n_all = len(pred_all)
    n_hit = int(pred_all["hit_all5"].sum())
    print("=" * 80)
    print("Coarse screening (ALL5) done.")
    print(f"Total rows merged: {n_all}")
    print(f"Hit ALL 5 conditions: {n_hit}")
    print("Outputs:", str(OUT_DIR))
    print("=" * 80)


if __name__ == "__main__":
    main()

Coarse screening (ALL5) done.
Total rows merged: 130
Hit ALL 5 conditions: 2
Outputs: C:\Users\86158\Desktop\coarse_screen_cat_sod_all5


In [8]:
# -*- coding: utf-8 -*-
"""
CAT + SOD 合并粗筛（5个条件全预测 + 统计每项命中数 + 导出 5/5 与 4/5）：

做什么：
1) 分别用 CAT/SOD 自己的数据全量重训你当时选好的 5 个分类模型（使用你给的最优参数）
2) 把 CAT+SOD 全部样本拼到一起，对每条样本都预测 5 个条件的概率/标签
3) 输出：
   - 严格：5/5 全满足（hit_all5）
   - 4/5：恰好 4 个条件为1（hit_exact4_of5），并导出缺失项统计
   - 宽松：CAT两项至少1项>=0.5 且 SOD三项至少1项>=0.5（hit_relaxed_cat1_sod1）
   - 同时给宽松候选做排序分数 score_relaxed 方便挑 Top-N
4) 终端打印：
   - 五项各自 pred==1 的数量（总体 + 按 CAT/SOD 分开）
   - 5/5、4/5、>=4/5 的数量
   - 4/5 时最常缺哪一项（计数）

输出文件：
OUT_DIR/
  - all_predictions_all5.csv
  - selected_all5.csv
  - selected_exact4_of5.csv
  - selected_exact4_of5_sorted.csv
  - selected_relaxed_cat1_sod1.csv
  - selected_relaxed_cat1_sod1_sorted.csv
  - coarse_screen_all5_config.json
  - models/
      cat_km_low_gb.joblib
      cat_vmax_high_dt.joblib
      sod_km_low_knn.joblib
      sod_vmax_high_xgb.joblib
      sod_ic50_low_xgb.joblib
"""

import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


# =========================
# 0) 路径 / 常量
# =========================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"   # <-- 改成你的
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"   # <-- 改成你的
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\coarse_screen_cat_sod_all5")  # <-- 改成你的
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

COL_DOI  = "data reference doi"
COL_KM   = "Km/mM"
COL_VMAX = "Vmax/μM s-1"
COL_IC50 = "IC50/μM"

# CAT 固定阈值（log10尺度，来自你CAT最优模型SHAP脚本）
CAT_KM_LO     = 1.0253058652647702
CAT_KM_HI     = 1.8055008581584002
CAT_VMAX_THR  = 0.06519065318914162

# SOD 固定阈值（来自你SOD IC50阈值脚本）
SOD_THR_IC50_UM   = 0.109
SOD_THR_IC50_UGML = 0.7

# 把概率转成 0/1 的阈值（仍然用0.5）
PROBA_THR = 0.5


# =========================
# 1) 特征定义（列名归一后使用）
# =========================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate"
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = ["shape", "surface treatment", "dispersion medium", "Substrate"]


# =========================
# 2) 通用工具：列名归一 + 特征准备
# =========================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def ensure_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in FEATURE_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    df_raw = ensure_feature_columns(df_raw)
    X = df_raw[FEATURE_COLS].copy()

    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        X[c] = X[c].fillna(0.0)

    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess_dense():
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop"
    )


# =========================
# 3) 标签工具
# =========================
def make_label_median_log10(series: pd.Series, mode: str):
    v = pd.to_numeric(series, errors="coerce")
    mask = v.notna() & (v > 0)
    vals = v.loc[mask].astype(float).values
    logv = np.log10(vals)
    thr = float(np.median(logv))
    if mode == "low_good":
        y = (logv <= thr).astype(int)
    else:
        y = (logv >= thr).astype(int)
    return mask, y, thr


def parse_ic50_cell(x):
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")
    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"
    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


def label_sod_ic50(ic50_value, ic50_unit):
    if pd.isna(ic50_value):
        return np.nan
    try:
        v = float(ic50_value)
    except Exception:
        return np.nan
    if v <= 0:
        return np.nan

    if ic50_unit == "uM":
        return int(v <= SOD_THR_IC50_UM)
    if ic50_unit == "ug/mL":
        return int(v <= SOD_THR_IC50_UGML)
    return np.nan


def safe_auc(y_true, p):
    try:
        if len(np.unique(y_true)) < 2:
            return None
        return float(roc_auc_score(y_true, p))
    except Exception:
        return None


# =========================
# 4) 训练你当时的“最优模型”
# =========================
def train_cat_models(df_cat: pd.DataFrame):
    preprocess_km = make_preprocess_dense()
    preprocess_v  = make_preprocess_dense()

    # KmLow extremes-only
    km = pd.to_numeric(df_cat.get(COL_KM), errors="coerce")
    m = km.notna() & (km > 0)
    df0 = df_cat.loc[m].copy()
    logkm = np.log10(km.loc[m].astype(float).values)

    keep = (logkm <= CAT_KM_LO) | (logkm >= CAT_KM_HI)
    df_train = df0.iloc[keep].copy()
    y_train = (logkm[keep] <= CAT_KM_LO).astype(int)

    km_model = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    )
    km_pipe = Pipeline([("preprocess", preprocess_km), ("model", km_model)])
    km_pipe.fit(prepare_features(df_train), y_train)
    km_auc = safe_auc(y_train, km_pipe.predict_proba(prepare_features(df_train))[:, 1])

    # VmaxHigh fixed threshold
    vmax = pd.to_numeric(df_cat.get(COL_VMAX), errors="coerce")
    mv = vmax.notna() & (vmax > 0)
    dfv = df_cat.loc[mv].copy()
    logv = np.log10(vmax.loc[mv].astype(float).values)
    yv = (logv >= CAT_VMAX_THR).astype(int)

    v_model = DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=2, class_weight="balanced", random_state=42
    )
    v_pipe = Pipeline([("preprocess", preprocess_v), ("model", v_model)])
    v_pipe.fit(prepare_features(dfv), yv)
    v_auc = safe_auc(yv, v_pipe.predict_proba(prepare_features(dfv))[:, 1])

    info = {
        "CAT_KmLow_train_extreme_n": int(len(df_train)),
        "CAT_KmLow_fixed_lo_log10": float(CAT_KM_LO),
        "CAT_KmLow_fixed_hi_log10": float(CAT_KM_HI),
        "CAT_KmLow_train_auc_sanity": km_auc,
        "CAT_VmaxHigh_train_n": int(len(dfv)),
        "CAT_VmaxHigh_fixed_thr_log10": float(CAT_VMAX_THR),
        "CAT_VmaxHigh_train_auc_sanity": v_auc,
    }
    return km_pipe, v_pipe, info


def train_sod_models(df_sod: pd.DataFrame):
    if not HAS_XGBOOST:
        raise RuntimeError("未检测到 xgboost，请先安装：pip install xgboost")

    preprocess_km = make_preprocess_dense()
    preprocess_v  = make_preprocess_dense()
    preprocess_ic = make_preprocess_dense()

    # KmLow median split
    m_km, y_km, thr_km = make_label_median_log10(df_sod.get(COL_KM), mode="low_good")
    df_km = df_sod.loc[m_km].copy()
    X_km = prepare_features(df_km)

    km_pipe = Pipeline([
        ("preprocess", preprocess_km),
        ("scaler", StandardScaler(with_mean=True)),
        ("model", KNeighborsClassifier(n_neighbors=5, weights="distance")),
    ])
    km_pipe.fit(X_km, y_km)
    km_auc = safe_auc(y_km, km_pipe.predict_proba(X_km)[:, 1])

    # VmaxHigh median split
    m_v, y_v, thr_v = make_label_median_log10(df_sod.get(COL_VMAX), mode="high_good")
    df_v = df_sod.loc[m_v].copy()
    X_v = prepare_features(df_v)

    v_model = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", random_state=42, n_jobs=-1, verbosity=0
    )
    v_pipe = Pipeline([("preprocess", preprocess_v), ("model", v_model)])
    v_pipe.fit(X_v, y_v)
    v_auc = safe_auc(y_v, v_pipe.predict_proba(X_v)[:, 1])

    # IC50Low fixed threshold by unit
    df2 = df_sod.copy()
    parsed = df2.get(COL_IC50).apply(parse_ic50_cell)
    df2["IC50_value"] = parsed.apply(lambda t: t[0])
    df2["IC50_unit"]  = parsed.apply(lambda t: t[1])
    df2["y_IC50Low"]   = df2.apply(lambda r: label_sod_ic50(r["IC50_value"], r["IC50_unit"]), axis=1)

    df_ic = df2[df2["y_IC50Low"].notna()].copy()
    X_ic = prepare_features(df_ic)
    y_ic = df_ic["y_IC50Low"].astype(int).values

    ic_model = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", random_state=42, n_jobs=-1, verbosity=0
    )
    ic_pipe = Pipeline([("preprocess", preprocess_ic), ("model", ic_model)])
    ic_pipe.fit(X_ic, y_ic)
    ic_auc = safe_auc(y_ic, ic_pipe.predict_proba(X_ic)[:, 1])

    info = {
        "SOD_KmLow_train_n": int(len(df_km)),
        "SOD_KmLow_median_thr_log10": float(thr_km),
        "SOD_KmLow_train_auc_sanity": km_auc,
        "SOD_VmaxHigh_train_n": int(len(df_v)),
        "SOD_VmaxHigh_median_thr_log10": float(thr_v),
        "SOD_VmaxHigh_train_auc_sanity": v_auc,
        "SOD_IC50Low_train_n": int(len(df_ic)),
        "SOD_IC50Low_fixed_thr_uM": float(SOD_THR_IC50_UM),
        "SOD_IC50Low_fixed_thr_ugmL": float(SOD_THR_IC50_UGML),
        "SOD_IC50Low_train_auc_sanity": ic_auc,
    }
    return km_pipe, v_pipe, ic_pipe, info


# =========================
# 5) 预测 + 导出规则
# =========================
COND_COLS = [
    "pred_CAT_KmLow",
    "pred_CAT_VmaxHigh",
    "pred_SOD_KmLow",
    "pred_SOD_VmaxHigh",
    "pred_SOD_IC50Low",
]

PROBA_COLS = [
    "p_CAT_KmLow",
    "p_CAT_VmaxHigh",
    "p_SOD_KmLow",
    "p_SOD_VmaxHigh",
    "p_SOD_IC50Low",
]


def predict_all5(df_all: pd.DataFrame,
                 cat_km_pipe, cat_v_pipe,
                 sod_km_pipe, sod_v_pipe, sod_ic_pipe) -> pd.DataFrame:
    X_all = prepare_features(df_all)

    p1 = cat_km_pipe.predict_proba(X_all)[:, 1]
    p2 = cat_v_pipe.predict_proba(X_all)[:, 1]
    p3 = sod_km_pipe.predict_proba(X_all)[:, 1]
    p4 = sod_v_pipe.predict_proba(X_all)[:, 1]
    p5 = sod_ic_pipe.predict_proba(X_all)[:, 1]

    out = df_all.copy()
    out["p_CAT_KmLow"] = p1
    out["p_CAT_VmaxHigh"] = p2
    out["p_SOD_KmLow"] = p3
    out["p_SOD_VmaxHigh"] = p4
    out["p_SOD_IC50Low"] = p5

    out["pred_CAT_KmLow"] = (p1 >= PROBA_THR).astype(int)
    out["pred_CAT_VmaxHigh"] = (p2 >= PROBA_THR).astype(int)
    out["pred_SOD_KmLow"] = (p3 >= PROBA_THR).astype(int)
    out["pred_SOD_VmaxHigh"] = (p4 >= PROBA_THR).astype(int)
    out["pred_SOD_IC50Low"] = (p5 >= PROBA_THR).astype(int)

    out["n_ones_5conds"] = out[COND_COLS].sum(axis=1)

    # 严格：5/5
    out["hit_all5"] = (out["n_ones_5conds"] == 5).astype(int)

    # 4/5：恰好4个1
    out["hit_exact4_of5"] = (out["n_ones_5conds"] == 4).astype(int)

    # 宽松：CAT两项至少1项>=0.5 且 SOD三项至少1项>=0.5
    out["cat_any_ge_0p5"] = (
        (out["p_CAT_KmLow"] >= 0.5) |
        (out["p_CAT_VmaxHigh"] >= 0.5)
    ).astype(int)

    out["sod_any_ge_0p5"] = (
        (out["p_SOD_KmLow"] >= 0.5) |
        (out["p_SOD_VmaxHigh"] >= 0.5) |
        (out["p_SOD_IC50Low"] >= 0.5)
    ).astype(int)

    out["hit_relaxed_cat1_sod1"] = (
        (out["cat_any_ge_0p5"] == 1) &
        (out["sod_any_ge_0p5"] == 1)
    ).astype(int)

    # 排序分数：CAT里最大概率 + SOD里最大概率
    out["score_relaxed"] = (
        out[["p_CAT_KmLow", "p_CAT_VmaxHigh"]].max(axis=1) +
        out[["p_SOD_KmLow", "p_SOD_VmaxHigh", "p_SOD_IC50Low"]].max(axis=1)
    )

    # 对于 4/5：也给一个排序分数（缺一项也能比较）
    out["score_exact4"] = (
        out[PROBA_COLS].mean(axis=1)
    )

    return out


def print_condition_counts(pred_all: pd.DataFrame):
    print("\n" + "=" * 90)
    print("Counts (overall) for each of the 5 conditions (pred==1):")
    for c in COND_COLS:
        print(f"  {c}: {int((pred_all[c] == 1).sum())} / {len(pred_all)}")

    n5 = int((pred_all["n_ones_5conds"] == 5).sum())
    n4 = int((pred_all["n_ones_5conds"] == 4).sum())
    nge4 = int((pred_all["n_ones_5conds"] >= 4).sum())
    print(f"\nExactly 5/5 ones: {n5} / {len(pred_all)}")
    print(f"Exactly 4/5 ones: {n4} / {len(pred_all)}")
    print(f">=4/5 ones:       {nge4} / {len(pred_all)}")

    # 4/5 时缺失项统计
    df4 = pred_all[pred_all["n_ones_5conds"] == 4].copy()
    if len(df4) > 0:
        miss_cond = df4[COND_COLS].eq(0).idxmax(axis=1)
        print("\nMissing-condition counts (within exactly-4):")
        print(miss_cond.value_counts().to_string())

    # 按数据集分开
    if "source_dataset" in pred_all.columns:
        print("\nCounts by source_dataset:")
        for ds, g in pred_all.groupby("source_dataset"):
            print(f"\n[{ds}] n={len(g)}")
            for c in COND_COLS:
                print(f"  {c}: {int((g[c] == 1).sum())} / {len(g)}")
            n5d = int((g["n_ones_5conds"] == 5).sum())
            n4d = int((g["n_ones_5conds"] == 4).sum())
            print(f"  exactly_5of5: {n5d} / {len(g)}")
            print(f"  exactly_4of5: {n4d} / {len(g)}")
            if "hit_relaxed_cat1_sod1" in g.columns:
                print(f"  relaxed_cat1_sod1: {int(g['hit_relaxed_cat1_sod1'].sum())} / {len(g)}")
    print("=" * 90 + "\n")


# =========================
# 6) 主程序
# =========================
def main():
    df_cat = normalize_columns(pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME))
    df_sod = normalize_columns(pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME))

    df_cat = df_cat.copy()
    df_sod = df_sod.copy()
    df_cat["source_dataset"] = "CAT"
    df_sod["source_dataset"] = "SOD"
    df_cat["orig_index"] = df_cat.index.values
    df_sod["orig_index"] = df_sod.index.values

    df_cat = ensure_feature_columns(df_cat)
    df_sod = ensure_feature_columns(df_sod)
    df_all = pd.concat([df_cat, df_sod], ignore_index=True)

    # train 5 models
    cat_km_pipe, cat_v_pipe, cat_info = train_cat_models(df_cat)
    sod_km_pipe, sod_v_pipe, sod_ic_pipe, sod_info = train_sod_models(df_sod)

    # save models
    joblib.dump(cat_km_pipe, OUT_DIR / "models" / "cat_km_low_gb.joblib")
    joblib.dump(cat_v_pipe,  OUT_DIR / "models" / "cat_vmax_high_dt.joblib")
    joblib.dump(sod_km_pipe, OUT_DIR / "models" / "sod_km_low_knn.joblib")
    joblib.dump(sod_v_pipe,  OUT_DIR / "models" / "sod_vmax_high_xgb.joblib")
    joblib.dump(sod_ic_pipe, OUT_DIR / "models" / "sod_ic50_low_xgb.joblib")

    # predict
    pred_all = predict_all5(df_all, cat_km_pipe, cat_v_pipe, sod_km_pipe, sod_v_pipe, sod_ic_pipe)

    # print counts
    print_condition_counts(pred_all)

    # export all
    pred_all.to_csv(OUT_DIR / "all_predictions_all5.csv", index=False, encoding="utf-8-sig")

    # export strict 5/5
    sel_all5 = pred_all[pred_all["hit_all5"] == 1].copy()
    sel_all5.to_csv(OUT_DIR / "selected_all5.csv", index=False, encoding="utf-8-sig")

    # export exactly 4/5 (new requirement)
    sel_4of5 = pred_all[pred_all["hit_exact4_of5"] == 1].copy()
    sel_4of5.to_csv(OUT_DIR / "selected_exact4_of5.csv", index=False, encoding="utf-8-sig")

    # sorted 4/5 by average proba
    sel_4of5_sorted = sel_4of5.sort_values("score_exact4", ascending=False)
    sel_4of5_sorted.to_csv(OUT_DIR / "selected_exact4_of5_sorted.csv", index=False, encoding="utf-8-sig")

    # export relaxed
    sel_relaxed = pred_all[pred_all["hit_relaxed_cat1_sod1"] == 1].copy()
    sel_relaxed.to_csv(OUT_DIR / "selected_relaxed_cat1_sod1.csv", index=False, encoding="utf-8-sig")

    sel_relaxed_sorted = sel_relaxed.sort_values("score_relaxed", ascending=False)
    sel_relaxed_sorted.to_csv(OUT_DIR / "selected_relaxed_cat1_sod1_sorted.csv", index=False, encoding="utf-8-sig")

    # config
    config = {
        "PROBA_THR": PROBA_THR,
        "CAT_fixed_thresholds_log10": {
            "KmLow_lo": CAT_KM_LO,
            "KmLow_hi": CAT_KM_HI,
            "VmaxHigh_thr": CAT_VMAX_THR
        },
        "SOD_fixed_thresholds": {
            "IC50_uM": SOD_THR_IC50_UM,
            "IC50_ugmL": SOD_THR_IC50_UGML
        },
        "train_sanity_auc": {**cat_info, **sod_info},
        "exports": [
            "all_predictions_all5.csv",
            "selected_all5.csv",
            "selected_exact4_of5.csv",
            "selected_exact4_of5_sorted.csv",
            "selected_relaxed_cat1_sod1.csv",
            "selected_relaxed_cat1_sod1_sorted.csv",
        ]
    }
    with open(OUT_DIR / "coarse_screen_all5_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    # summary
    n_all = len(pred_all)
    n5 = int(pred_all["hit_all5"].sum())
    n4 = int(pred_all["hit_exact4_of5"].sum())
    nrel = int(pred_all["hit_relaxed_cat1_sod1"].sum())
    print("=" * 90)
    print("Done.")
    print(f"Total merged rows: {n_all}")
    print(f"Hit strict ALL5: {n5}")
    print(f"Hit exactly 4/5: {n4}")
    print(f"Hit relaxed (CAT>=1 & SOD>=1): {nrel}")
    print("Outputs:", str(OUT_DIR))
    print("=" * 90)


if __name__ == "__main__":
    main()


Counts (overall) for each of the 5 conditions (pred==1):
  pred_CAT_KmLow: 61 / 130
  pred_CAT_VmaxHigh: 70 / 130
  pred_SOD_KmLow: 76 / 130
  pred_SOD_VmaxHigh: 52 / 130
  pred_SOD_IC50Low: 68 / 130

Exactly 5/5 ones: 2 / 130
Exactly 4/5 ones: 26 / 130
>=4/5 ones:       28 / 130

Missing-condition counts (within exactly-4):
pred_CAT_VmaxHigh    13
pred_SOD_IC50Low      5
pred_CAT_KmLow        4
pred_SOD_VmaxHigh     3
pred_SOD_KmLow        1

Counts by source_dataset:

[CAT] n=70
  pred_CAT_KmLow: 31 / 70
  pred_CAT_VmaxHigh: 39 / 70
  pred_SOD_KmLow: 44 / 70
  pred_SOD_VmaxHigh: 29 / 70
  pred_SOD_IC50Low: 39 / 70
  exactly_5of5: 2 / 70
  exactly_4of5: 17 / 70
  relaxed_cat1_sod1: 43 / 70

[SOD] n=60
  pred_CAT_KmLow: 30 / 60
  pred_CAT_VmaxHigh: 31 / 60
  pred_SOD_KmLow: 32 / 60
  pred_SOD_VmaxHigh: 23 / 60
  pred_SOD_IC50Low: 29 / 60
  exactly_5of5: 0 / 60
  exactly_4of5: 9 / 60
  relaxed_cat1_sod1: 43 / 60

Done.
Total merged rows: 130
Hit strict ALL5: 2
Hit exactly 4/5: 26
Hit r

In [1]:
# -*- coding: utf-8 -*-
"""
Refit 4 best models (full-fit) using best params, then score & rank 26+2 candidates.

Weights:
- CAT total weight = 0.5
- SOD total weight = 0.5, split equally across 3 models => each SOD model weight = 1/6

Scoring:
- Convert each model output to percentile score in [0, 1] on candidate set
- Direction:
    IC50: lower is better -> use -pred_logIC50
    Km:   lower is better -> use -pred_logKm
    Kcat: higher is better -> use +pred_logKcat
"""

import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import VarianceThreshold, f_regression

from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge


# =========================
# 0) PATHS (EDIT HERE)
# =========================
SOD_XLSX = Path(r"C:\Users\86158\Desktop\dataissod.xlsx")
CAT_XLSX = Path(r"C:\Users\86158\Desktop\dataiscat.xlsx")
SHEET_NAME = 0

CANDIDATE_CSVS = [
    Path(r"C:\Users\86158\Desktop\26.csv"),
    Path(r"C:\Users\86158\Desktop\2.csv"),
]

OUT_CSV  = Path(r"C:\Users\86158\Desktop\ranked_28_weighted.csv")
OUT_XLSX = Path(r"C:\Users\86158\Desktop\ranked_28_weighted.xlsx")

# =========================
# 1) Column names & feature list (canonical after normalize_columns)
# =========================
COL_DOI = "data reference doi"
COL_IC50 = "IC50/μM"
COL_KM = "Km/mM"
COL_KCAT = "Kcat/s-1"

FEATURE_COLS = [
    "contain O","contain N","contain P","contain S","contain Si",
    "contain Se","contain B","contain F","contain Cl","contain Br","contain I",
    "main metal proportion","main metal number","main metal valence",
    "minor metal percentage","minor metal number","minor metal valence",
    "3rd metal ratio","3rd metal type","3rd metal valence",
    "shape","size (nm)","surface treatment","dispersion medium",
    "pH","temperature/oC","Substrate"
]
NUMERIC_COLS = [
    "contain O","contain N","contain P","contain S","contain Si",
    "contain Se","contain B","contain F","contain Cl","contain Br","contain I",
    "main metal proportion","main metal number","main metal valence",
    "minor metal percentage","minor metal number","minor metal valence",
    "3rd metal ratio","3rd metal type","3rd metal valence",
    "size (nm)","pH","temperature/oC"
]
CATEGORICAL_COLS = ["shape","surface treatment","dispersion medium","Substrate"]


# =========================
# 2) Best configs (from your screenshots)
# =========================
BEST_SOD_IC50_UGML = dict(
    name="SOD_logIC50_ugmL",
    dataset="SOD",
    task="ic50_ugml",
    preprocess=dict(min_freq_cat=3, var_thr=0.0, k_best=None, svd_dim=10),
    model=dict(type="SVR_RBF", params=dict(C=30, gamma=0.01, epsilon=0.2)),
    direction="lower_better",  # IC50 lower better
)

BEST_SOD_LOGKM = dict(
    name="SOD_logKm",
    dataset="SOD",
    task="logkm",
    preprocess=dict(min_freq_cat=2, var_thr=0.0, k_best=None, svd_dim=8),
    model=dict(type="SVR_RBF", params=dict(C=100, gamma=0.03, epsilon=0.03)),
    direction="lower_better",  # Km lower better
)

BEST_SOD_LOGKCAT = dict(
    name="SOD_logKcat",
    dataset="SOD",
    task="logkcat",
    preprocess=dict(min_freq_cat=2, var_thr=0.0, k_best=12, svd_dim=None),
    model=dict(type="KernelRidge_RBF", params=dict(alpha=0.01, gamma=0.01)),
    direction="higher_better",  # Kcat higher better
)

BEST_CAT_LOGKCAT = dict(
    name="CAT_logKcat",
    dataset="CAT",
    task="logkcat",
    preprocess=dict(min_freq_cat=2, var_thr=0.0, k_best=15, svd_dim=6),
    model=dict(type="KernelRidge_RBF", params=dict(alpha=0.01, gamma=0.03)),
    direction="higher_better",  # Kcat higher better
)

MODEL_SPECS = [BEST_SOD_IC50_UGML, BEST_SOD_LOGKM, BEST_SOD_LOGKCAT, BEST_CAT_LOGKCAT]


# =========================
# 3) Utilities
# =========================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    # require all features
    missing = [c for c in FEATURE_COLS if c not in df_raw.columns]
    if missing:
        raise ValueError(f"Missing feature columns: {missing}")

    X = df_raw[FEATURE_COLS].copy()

    for c in NUMERIC_COLS:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    # keep your earlier convention: 3rd metal missing -> 0
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in CATEGORICAL_COLS:
        if c in X.columns:
            X[c] = clean_text_series(X[c])

    return X


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """Fold-wise rare category grouping: freq < min_freq -> __other__"""
    def __init__(self, min_freq=3, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


def build_preprocess(min_freq_cat=3, sparse=True):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=sparse)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=min_freq_cat)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop",
    )


class ToDense(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return np.asarray(X)


class SelectKBestSafe(BaseEstimator, TransformerMixin):
    def __init__(self, k=None):
        self.k = k
        self.selector_ = None

    def fit(self, X, y):
        if self.k is None:
            self.selector_ = None
            return self
        n_feat = X.shape[1]
        k_eff = int(min(int(self.k), int(n_feat)))
        if k_eff <= 0:
            self.selector_ = None
            return self
        from sklearn.feature_selection import SelectKBest
        self.selector_ = SelectKBest(score_func=f_regression, k=k_eff)
        self.selector_.fit(X, y)
        return self

    def transform(self, X):
        if self.selector_ is None:
            return X
        return self.selector_.transform(X)


class TruncatedSVDSafe(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=None, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.svd_ = None

    def fit(self, X, y=None):
        if self.n_components is None:
            self.svd_ = None
            return self
        n_feat = X.shape[1]
        if n_feat <= 1:
            self.svd_ = None
            return self
        n_eff = min(int(self.n_components), max(1, n_feat - 1))
        self.svd_ = TruncatedSVD(n_components=n_eff, random_state=self.random_state)
        self.svd_.fit(X)
        return self

    def transform(self, X):
        if self.svd_ is None:
            return X
        return self.svd_.transform(X)


def build_pipeline_from_spec(spec: dict, random_seed=42) -> Pipeline:
    pre = spec["preprocess"]
    model = spec["model"]

    model_name = model["type"]
    model_params = model["params"]

    # which models need dense
    needs_dense = model_name in {"SVR_RBF", "KernelRidge_RBF"}

    steps = []
    steps.append(("preprocess", build_preprocess(min_freq_cat=pre["min_freq_cat"], sparse=True)))
    steps.append(("var", VarianceThreshold(threshold=float(pre["var_thr"]))))

    if pre["k_best"] is not None:
        steps.append(("kbest", SelectKBestSafe(k=int(pre["k_best"]))))

    if pre["svd_dim"] is not None:
        steps.append(("svd", TruncatedSVDSafe(n_components=int(pre["svd_dim"]), random_state=random_seed)))
        steps.append(("scaler", StandardScaler(with_mean=True)))
    else:
        if needs_dense:
            steps.append(("todense", ToDense()))
            steps.append(("scaler", StandardScaler(with_mean=True)))
        else:
            steps.append(("scaler", StandardScaler(with_mean=False)))

    if model_name == "SVR_RBF":
        est = SVR(kernel="rbf", **model_params)
    elif model_name == "KernelRidge_RBF":
        est = KernelRidge(kernel="rbf", **model_params)
    else:
        raise ValueError(f"Unknown model type: {model_name}")

    steps.append(("model", est))
    return Pipeline(steps)


def percentile_score(values: np.ndarray) -> np.ndarray:
    """Rank-percentile in [0,1], higher value => higher score."""
    v = pd.Series(values).astype(float)
    r = v.rank(method="average", ascending=True)
    if len(v) <= 1:
        return np.ones(len(v), dtype=float)
    return ((r - 1.0) / (len(v) - 1.0)).values


def parse_ic50_cell(x):
    """
    Parse IC50 cell:
      - pure number -> default uM
      - 'value + uM/μM/µM' -> uM
      - 'value + ug/mL/μg/mL/µg/mL' -> ug/mL
    Return: (value, unit)
    """
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")
    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"
    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


def load_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}")
    df = pd.read_excel(str(path), sheet_name=SHEET_NAME)
    return normalize_columns(df)


def load_candidates(csv_paths) -> pd.DataFrame:
    dfs = []
    for p in csv_paths:
        if not Path(p).exists():
            raise FileNotFoundError(f"Candidate file not found: {p}")
        df = pd.read_csv(p, encoding="utf-8-sig")
        df = normalize_columns(df)
        df["__source_file__"] = Path(p).name
        dfs.append(df)
    out = pd.concat(dfs, ignore_index=True)
    # drop unnamed index cols if exist
    for c in list(out.columns):
        if c.lower().startswith("unnamed:"):
            pass
    return out


# =========================
# 4) Fit each best model (full-fit)
# =========================
def fit_model_full(spec: dict, df_sod: pd.DataFrame, df_cat: pd.DataFrame) -> Pipeline:
    pipe = build_pipeline_from_spec(spec, random_seed=42)

    if spec["dataset"] == "SOD":
        df = df_sod.copy()
    else:
        df = df_cat.copy()

    # build y and filter
    if spec["task"] == "ic50_ugml":
        if COL_IC50 not in df.columns:
            raise KeyError(f"Missing IC50 column in {spec['dataset']}: {COL_IC50}")
        parsed = df[COL_IC50].apply(parse_ic50_cell)
        df["IC50_value"] = parsed.apply(lambda t: t[0])
        df["IC50_unit"] = parsed.apply(lambda t: t[1])
        df["IC50_value"] = pd.to_numeric(df["IC50_value"], errors="coerce")

        df_use = df[(df["IC50_unit"] == "ug/mL") & df["IC50_value"].notna() & (df["IC50_value"] > 0)].copy()
        if df_use.empty:
            raise RuntimeError("No valid ug/mL IC50 rows for training.")
        y = np.log10(df_use["IC50_value"].astype(float).values)

    elif spec["task"] == "logkm":
        if COL_KM not in df.columns:
            raise KeyError(f"Missing Km column in {spec['dataset']}: {COL_KM}")
        v = pd.to_numeric(df[COL_KM], errors="coerce")
        df_use = df[v.notna() & (v > 0)].copy()
        if df_use.empty:
            raise RuntimeError("No valid Km rows for training.")
        y = np.log10(pd.to_numeric(df_use[COL_KM], errors="coerce").astype(float).values)

    elif spec["task"] == "logkcat":
        if COL_KCAT not in df.columns:
            raise KeyError(f"Missing Kcat column in {spec['dataset']}: {COL_KCAT}")
        v = pd.to_numeric(df[COL_KCAT], errors="coerce")
        df_use = df[v.notna() & (v > 0)].copy()
        if df_use.empty:
            raise RuntimeError("No valid Kcat rows for training.")
        y = np.log10(pd.to_numeric(df_use[COL_KCAT], errors="coerce").astype(float).values)

    else:
        raise ValueError(f"Unknown task: {spec['task']}")

    X = prepare_features(df_use)

    pipe.fit(X, y)
    return pipe


# =========================
# 5) Predict candidates + weighted ranking
# =========================
def main():
    # load training data
    df_sod = load_dataset(SOD_XLSX)
    df_cat = load_dataset(CAT_XLSX)

    # load candidates (26+2)
    cand = load_candidates(CANDIDATE_CSVS)
    # ensure features exist
    _ = prepare_features(cand)  # will raise if missing
    X_cand = prepare_features(cand)

    # fit 4 models (full)
    fitted = {}
    for spec in MODEL_SPECS:
        print(f"[Fit full] {spec['name']}  ({spec['dataset']})")
        fitted[spec["name"]] = fit_model_full(spec, df_sod, df_cat)

    # predict
    # (all are log10 targets)
    cand["pred_SOD_logIC50_ugmL"] = fitted["SOD_logIC50_ugmL"].predict(X_cand)
    cand["pred_SOD_logKm"]        = fitted["SOD_logKm"].predict(X_cand)
    cand["pred_SOD_logKcat"]      = fitted["SOD_logKcat"].predict(X_cand)
    cand["pred_CAT_logKcat"]      = fitted["CAT_logKcat"].predict(X_cand)

    # direction -> higher is better
    v_ic50 = -cand["pred_SOD_logIC50_ugmL"].astype(float).values  # lower better
    v_km   = -cand["pred_SOD_logKm"].astype(float).values        # lower better
    v_sod_kcat = cand["pred_SOD_logKcat"].astype(float).values   # higher better
    v_cat_kcat = cand["pred_CAT_logKcat"].astype(float).values   # higher better

    # percentile scores (0..1)
    cand["score_ic50"] = percentile_score(v_ic50)
    cand["score_km"] = percentile_score(v_km)
    cand["score_sod_kcat"] = percentile_score(v_sod_kcat)
    cand["score_cat_kcat"] = percentile_score(v_cat_kcat)

    # weights
    w_sod_each = 0.5 / 3.0   # 1/6
    w_cat = 0.5

    cand["score_total"] = (
        w_sod_each * cand["score_ic50"]
        + w_sod_each * cand["score_km"]
        + w_sod_each * cand["score_sod_kcat"]
        + w_cat * cand["score_cat_kcat"]
    )

    cand["rank"] = cand["score_total"].rank(method="first", ascending=False).astype(int)
    cand = cand.sort_values("rank", ascending=True).reset_index(drop=True)

    # save
    # keep some useful columns if present
    show_cols = [
        "rank","nanozyme","classification","__source_file__", COL_DOI,
        "score_total","score_cat_kcat","score_ic50","score_km","score_sod_kcat",
        "pred_SOD_logIC50_ugmL","pred_SOD_logKm","pred_SOD_logKcat","pred_CAT_logKcat",
    ]
    show_cols = [c for c in show_cols if c in cand.columns]

    cand[show_cols].to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    cand[show_cols].to_excel(OUT_XLSX, index=False)

    print("\n[Saved]", OUT_CSV)
    print("[Saved]", OUT_XLSX)
    print("\nTop 10 preview:")
    print(cand[show_cols].head(10).to_string(index=False))


if __name__ == "__main__":
    main()

[Fit full] SOD_logIC50_ugmL  (SOD)
[Fit full] SOD_logKm  (SOD)
[Fit full] SOD_logKcat  (SOD)
[Fit full] CAT_logKcat  (CAT)

[Saved] C:\Users\86158\Desktop\ranked_28_weighted.csv
[Saved] C:\Users\86158\Desktop\ranked_28_weighted.xlsx

Top 10 preview:
 rank               nanozyme classification __source_file__         data reference doi  score_total  score_cat_kcat  score_ic50  score_km  score_sod_kcat  pred_SOD_logIC50_ugmL  pred_SOD_logKm  pred_SOD_logKcat  pred_CAT_logKcat
    1                AuPt3Cu            CAT          26.csv  10.1016/j.cej.2023.142494     0.790123        0.962963    0.481481  0.703704        0.666667              -1.272225       -0.577475          1.439859          7.284993
    2                  AuPt3            CAT          26.csv  10.1016/j.cej.2023.142494     0.759259        1.000000    0.444444  0.555556        0.555556              -1.197093       -0.113182          1.384653          7.672220
    3 CeO2@MPEG2KMPEGa1K-MPh            CAT          26.csv    

In [4]:
# -*- coding: utf-8 -*-
"""
CAT + SOD 六指标联合粗筛
- 训练集：CAT / SOD
- 预测对象：CAT + SOD + POD + OXD 全部样本
- CAT三项：KmLow / VmaxHigh / KcatHigh
- SOD三项：KmLow / VmaxHigh / IC50Low

统计规则：
在 “CAT三项至少1项为1 且 SOD三项至少1项为1” 的基础上，
统计 6/6、5/6、4/6、3/6、2/6 的数量

输出：
1) all_predictions_all6.csv
   保留原表所有列，只在最后新增：
   - 6个概率列
   - 6个预测标签列
   - CAT/SOD/ALL 的命中数
   - base条件与 6/6~2/6 命中列
2) summary_counts_all6.csv
3) coarse_screen_all6_config.json
4) models/*.joblib
"""

import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


# =========================================================
# 0) 路径 / 常量
# =========================================================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"
POD_PATH = r"C:\Users\86158\Desktop\dataispod.xlsx"
OXD_PATH = r"C:\Users\86158\Desktop\dataisoxd.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\coarse_screen_cat_sod_all6")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
PROBA_THR = 0.5

COL_DOI  = "data reference doi"
COL_KM   = "Km/mM"
COL_VMAX = "Vmax/μM s-1"
COL_KCAT = "Kcat/s-1"
COL_IC50 = "IC50/μM"

# -------------------------
# CAT 固定阈值（你之前已经确定）
# -------------------------
CAT_KM_LO    = 1.0253058652647702
CAT_KM_HI    = 1.8055008581584002
CAT_VMAX_THR = 0.06519065318914162

# -------------------------
# CAT KcatHigh（按你刚给的脚本）
# -------------------------
CAT_KCAT_SCHEME = "single_q"
CAT_KCAT_Q = 0.45
CAT_KCAT_EXPECTED_THR = 3.3198654689156797  # sanity check only

# -------------------------
# SOD 固定阈值
# -------------------------
SOD_THR_IC50_UM   = 0.109
SOD_THR_IC50_UGML = 0.7


# =========================================================
# 1) 特征定义
# 说明：
# 你之前不同脚本里出现过：
# - "main  metal proportion"（双空格）
# - "Substrate "（尾空格）
# 这里先统一列名，再用规范名。
# =========================================================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate"
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = [
    "shape", "surface treatment", "dispersion medium", "Substrate"
]


# =========================================================
# 2) 工具函数
# =========================================================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def ensure_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in FEATURE_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    df_raw = ensure_feature_columns(df_raw)
    X = df_raw[FEATURE_COLS].copy()

    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0.0)

    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess_dense():
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop"
    )


def make_label_median_log10(series: pd.Series, mode: str):
    v = pd.to_numeric(series, errors="coerce")
    mask = v.notna() & (v > 0)
    vals = v.loc[mask].astype(float).values
    logv = np.log10(vals)
    thr = float(np.median(logv))

    if mode == "low_good":
        y = (logv <= thr).astype(int)
    else:
        y = (logv >= thr).astype(int)

    return mask, y, thr


def build_labels_log10_single_q(values: pd.Series, task: str, q: float):
    """
    完全复用你 Kcat 脚本的标签思路：
    - 先取正值并 log10
    - 若 task != KmLow，则阈值取 quantile(1-q)
    - High任务：y = logv >= thr
    返回：
      idx_keep, y, meta
    """
    v = pd.to_numeric(values, errors="coerce")
    m = v.notna() & (v > 0)
    vals = np.log10(v.loc[m].astype(float).values)

    if len(vals) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), {
            "scheme": "single_q", "q": q, "thr": np.nan
        }

    if task == "KmLow":
        thr = float(np.quantile(vals, q))
        y = (vals <= thr).astype(int)
    else:
        thr = float(np.quantile(vals, 1 - q))
        y = (vals >= thr).astype(int)

    idx_all = np.where(m.values)[0]
    meta = {"scheme": "single_q", "q": q, "thr": thr}
    return idx_all, y.astype(int), meta


def parse_ic50_cell(x):
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")
    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"
    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


def label_sod_ic50(ic50_value, ic50_unit):
    if pd.isna(ic50_value):
        return np.nan
    try:
        v = float(ic50_value)
    except Exception:
        return np.nan
    if v <= 0:
        return np.nan

    if ic50_unit == "uM":
        return int(v <= SOD_THR_IC50_UM)
    if ic50_unit == "ug/mL":
        return int(v <= SOD_THR_IC50_UGML)
    return np.nan


def safe_auc(y_true, p):
    try:
        if len(np.unique(y_true)) < 2:
            return None
        return float(roc_auc_score(y_true, p))
    except Exception:
        return None


# =========================================================
# 3) 训练模型
# =========================================================
def train_cat_models(df_cat: pd.DataFrame):
    preprocess_km = make_preprocess_dense()
    preprocess_v  = make_preprocess_dense()
    preprocess_kc = make_preprocess_dense()

    # -----------------------------------------------------
    # CAT KmLow
    # extremes-only + GradientBoosting
    # -----------------------------------------------------
    km = pd.to_numeric(df_cat.get(COL_KM), errors="coerce")
    m = km.notna() & (km > 0)
    df0 = df_cat.loc[m].copy()
    logkm = np.log10(km.loc[m].astype(float).values)

    keep = (logkm <= CAT_KM_LO) | (logkm >= CAT_KM_HI)
    df_train_km = df0.iloc[keep].copy()
    y_train_km = (logkm[keep] <= CAT_KM_LO).astype(int)

    km_model = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_SEED
    )
    km_pipe = Pipeline([
        ("preprocess", preprocess_km),
        ("model", km_model)
    ])
    X_train_km = prepare_features(df_train_km)
    km_pipe.fit(X_train_km, y_train_km)
    km_auc = safe_auc(y_train_km, km_pipe.predict_proba(X_train_km)[:, 1])

    # -----------------------------------------------------
    # CAT VmaxHigh
    # fixed threshold + DecisionTree
    # -----------------------------------------------------
    vmax = pd.to_numeric(df_cat.get(COL_VMAX), errors="coerce")
    mv = vmax.notna() & (vmax > 0)
    df_train_v = df_cat.loc[mv].copy()
    logv = np.log10(vmax.loc[mv].astype(float).values)
    y_train_v = (logv >= CAT_VMAX_THR).astype(int)

    v_model = DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_SEED
    )
    v_pipe = Pipeline([
        ("preprocess", preprocess_v),
        ("model", v_model)
    ])
    X_train_v = prepare_features(df_train_v)
    v_pipe.fit(X_train_v, y_train_v)
    v_auc = safe_auc(y_train_v, v_pipe.predict_proba(X_train_v)[:, 1])

    # -----------------------------------------------------
    # CAT KcatHigh
    # 完全按你刚给的脚本思路：
    # single_q(q=0.45) on log10(Kcat)
    # KNN(n_neighbors=5, weights='distance') + StandardScaler
    # -----------------------------------------------------
    idx_keep_kc, y_train_kc, meta_kc = build_labels_log10_single_q(
        df_cat[COL_KCAT],
        task="KcatHigh",
        q=CAT_KCAT_Q
    )

    if len(idx_keep_kc) == 0:
        raise RuntimeError("CAT KcatHigh 无可用于训练的有效样本（Kcat<=0 或缺失）")

    df_train_kc = df_cat.iloc[idx_keep_kc].copy()
    thr_kc = float(meta_kc["thr"])

    kc_model = KNeighborsClassifier(
        n_neighbors=5,
        weights="distance"
    )
    kc_pipe = Pipeline([
        ("preprocess", preprocess_kc),
        ("scaler", StandardScaler(with_mean=True)),
        ("model", kc_model)
    ])

    X_train_kc = prepare_features(df_train_kc)
    kc_pipe.fit(X_train_kc, y_train_kc)
    kc_auc = safe_auc(y_train_kc, kc_pipe.predict_proba(X_train_kc)[:, 1])

    info = {
        "CAT_KmLow_train_n": int(len(df_train_km)),
        "CAT_KmLow_fixed_lo_log10": float(CAT_KM_LO),
        "CAT_KmLow_fixed_hi_log10": float(CAT_KM_HI),
        "CAT_KmLow_model": "GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)",
        "CAT_KmLow_train_auc_sanity": km_auc,

        "CAT_VmaxHigh_train_n": int(len(df_train_v)),
        "CAT_VmaxHigh_fixed_thr_log10": float(CAT_VMAX_THR),
        "CAT_VmaxHigh_model": 'DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, class_weight="balanced", random_state=42)',
        "CAT_VmaxHigh_train_auc_sanity": v_auc,

        "CAT_KcatHigh_train_n": int(len(df_train_kc)),
        "CAT_KcatHigh_scheme": CAT_KCAT_SCHEME,
        "CAT_KcatHigh_q": CAT_KCAT_Q,
        "CAT_KcatHigh_thr_log10_from_refit": thr_kc,
        "CAT_KcatHigh_thr_log10_expected": float(CAT_KCAT_EXPECTED_THR),
        "CAT_KcatHigh_model": 'KNeighborsClassifier(n_neighbors=5, weights="distance") + StandardScaler',
        "CAT_KcatHigh_train_auc_sanity": kc_auc,
    }

    return km_pipe, v_pipe, kc_pipe, info


def train_sod_models(df_sod: pd.DataFrame):
    if not HAS_XGBOOST:
        raise RuntimeError("未检测到 xgboost，请先安装：pip install xgboost")

    preprocess_km = make_preprocess_dense()
    preprocess_v  = make_preprocess_dense()
    preprocess_ic = make_preprocess_dense()

    # -----------------------------------------------------
    # SOD KmLow
    # median split + KNN + StandardScaler
    # -----------------------------------------------------
    m_km, y_km, thr_km = make_label_median_log10(df_sod.get(COL_KM), mode="low_good")
    df_km = df_sod.loc[m_km].copy()
    X_km = prepare_features(df_km)

    km_pipe = Pipeline([
        ("preprocess", preprocess_km),
        ("scaler", StandardScaler(with_mean=True)),
        ("model", KNeighborsClassifier(n_neighbors=5, weights="distance")),
    ])
    km_pipe.fit(X_km, y_km)
    km_auc = safe_auc(y_km, km_pipe.predict_proba(X_km)[:, 1])

    # -----------------------------------------------------
    # SOD VmaxHigh
    # median split + XGB
    # -----------------------------------------------------
    m_v, y_v, thr_v = make_label_median_log10(df_sod.get(COL_VMAX), mode="high_good")
    df_v = df_sod.loc[m_v].copy()
    X_v = prepare_features(df_v)

    v_model = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=0
    )
    v_pipe = Pipeline([
        ("preprocess", preprocess_v),
        ("model", v_model)
    ])
    v_pipe.fit(X_v, y_v)
    v_auc = safe_auc(y_v, v_pipe.predict_proba(X_v)[:, 1])

    # -----------------------------------------------------
    # SOD IC50Low
    # fixed threshold by unit + XGB
    # -----------------------------------------------------
    df2 = df_sod.copy()
    parsed = df2.get(COL_IC50).apply(parse_ic50_cell)
    df2["IC50_value"] = parsed.apply(lambda t: t[0])
    df2["IC50_unit"]  = parsed.apply(lambda t: t[1])
    df2["y_IC50Low"]  = df2.apply(
        lambda r: label_sod_ic50(r["IC50_value"], r["IC50_unit"]), axis=1
    )

    df_ic = df2[df2["y_IC50Low"].notna()].copy()
    X_ic = prepare_features(df_ic)
    y_ic = df_ic["y_IC50Low"].astype(int).values

    ic_model = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=0
    )
    ic_pipe = Pipeline([
        ("preprocess", preprocess_ic),
        ("model", ic_model)
    ])
    ic_pipe.fit(X_ic, y_ic)
    ic_auc = safe_auc(y_ic, ic_pipe.predict_proba(X_ic)[:, 1])

    info = {
        "SOD_KmLow_train_n": int(len(df_km)),
        "SOD_KmLow_median_thr_log10": float(thr_km),
        "SOD_KmLow_model": 'KNeighborsClassifier(n_neighbors=5, weights="distance") + StandardScaler',
        "SOD_KmLow_train_auc_sanity": km_auc,

        "SOD_VmaxHigh_train_n": int(len(df_v)),
        "SOD_VmaxHigh_median_thr_log10": float(thr_v),
        "SOD_VmaxHigh_model": 'XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss", tree_method="hist", random_state=42, n_jobs=-1, verbosity=0)',
        "SOD_VmaxHigh_train_auc_sanity": v_auc,

        "SOD_IC50Low_train_n": int(len(df_ic)),
        "SOD_IC50Low_fixed_thr_uM": float(SOD_THR_IC50_UM),
        "SOD_IC50Low_fixed_thr_ugmL": float(SOD_THR_IC50_UGML),
        "SOD_IC50Low_model": 'XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss", tree_method="hist", random_state=42, n_jobs=-1, verbosity=0)',
        "SOD_IC50Low_train_auc_sanity": ic_auc,
    }

    return km_pipe, v_pipe, ic_pipe, info


# =========================================================
# 4) 全量预测 + 统计
# =========================================================
CAT_COND_COLS = [
    "pred_CAT_KmLow",
    "pred_CAT_VmaxHigh",
    "pred_CAT_KcatHigh",
]

SOD_COND_COLS = [
    "pred_SOD_KmLow",
    "pred_SOD_VmaxHigh",
    "pred_SOD_IC50Low",
]

ALL6_COND_COLS = CAT_COND_COLS + SOD_COND_COLS

ALL6_PROBA_COLS = [
    "p_CAT_KmLow",
    "p_CAT_VmaxHigh",
    "p_CAT_KcatHigh",
    "p_SOD_KmLow",
    "p_SOD_VmaxHigh",
    "p_SOD_IC50Low",
]


def predict_all6(df_all: pd.DataFrame,
                 cat_km_pipe, cat_v_pipe, cat_kc_pipe,
                 sod_km_pipe, sod_v_pipe, sod_ic_pipe) -> pd.DataFrame:
    X_all = prepare_features(df_all)

    p_cat_km = cat_km_pipe.predict_proba(X_all)[:, 1]
    p_cat_v  = cat_v_pipe.predict_proba(X_all)[:, 1]
    p_cat_kc = cat_kc_pipe.predict_proba(X_all)[:, 1]

    p_sod_km = sod_km_pipe.predict_proba(X_all)[:, 1]
    p_sod_v  = sod_v_pipe.predict_proba(X_all)[:, 1]
    p_sod_ic = sod_ic_pipe.predict_proba(X_all)[:, 1]

    out = df_all.copy()

    # 概率列
    out["p_CAT_KmLow"] = p_cat_km
    out["p_CAT_VmaxHigh"] = p_cat_v
    out["p_CAT_KcatHigh"] = p_cat_kc
    out["p_SOD_KmLow"] = p_sod_km
    out["p_SOD_VmaxHigh"] = p_sod_v
    out["p_SOD_IC50Low"] = p_sod_ic

    # 标签列
    out["pred_CAT_KmLow"] = (p_cat_km >= PROBA_THR).astype(int)
    out["pred_CAT_VmaxHigh"] = (p_cat_v >= PROBA_THR).astype(int)
    out["pred_CAT_KcatHigh"] = (p_cat_kc >= PROBA_THR).astype(int)

    out["pred_SOD_KmLow"] = (p_sod_km >= PROBA_THR).astype(int)
    out["pred_SOD_VmaxHigh"] = (p_sod_v >= PROBA_THR).astype(int)
    out["pred_SOD_IC50Low"] = (p_sod_ic >= PROBA_THR).astype(int)

    # 汇总列
    out["n_pos_CAT_3"] = out[CAT_COND_COLS].sum(axis=1)
    out["n_pos_SOD_3"] = out[SOD_COND_COLS].sum(axis=1)
    out["n_pos_ALL_6"] = out[ALL6_COND_COLS].sum(axis=1)

    out["CAT_any1"] = (out["n_pos_CAT_3"] >= 1).astype(int)
    out["SOD_any1"] = (out["n_pos_SOD_3"] >= 1).astype(int)

    out["base_CAT1_and_SOD1"] = (
        (out["CAT_any1"] == 1) &
        (out["SOD_any1"] == 1)
    ).astype(int)

    out["hit_6of6_under_base"] = (
        (out["base_CAT1_and_SOD1"] == 1) &
        (out["n_pos_ALL_6"] == 6)
    ).astype(int)

    out["hit_5of6_under_base"] = (
        (out["base_CAT1_and_SOD1"] == 1) &
        (out["n_pos_ALL_6"] == 5)
    ).astype(int)

    out["hit_4of6_under_base"] = (
        (out["base_CAT1_and_SOD1"] == 1) &
        (out["n_pos_ALL_6"] == 4)
    ).astype(int)

    out["hit_3of6_under_base"] = (
        (out["base_CAT1_and_SOD1"] == 1) &
        (out["n_pos_ALL_6"] == 3)
    ).astype(int)

    out["hit_2of6_under_base"] = (
        (out["base_CAT1_and_SOD1"] == 1) &
        (out["n_pos_ALL_6"] == 2)
    ).astype(int)

    out["mean_proba_ALL6"] = out[ALL6_PROBA_COLS].mean(axis=1)

    return out


def build_summary_table(pred_all: pd.DataFrame) -> pd.DataFrame:
    def summarize_scope(df_scope: pd.DataFrame, scope_name: str) -> dict:
        n_total = len(df_scope)
        n_base = int(df_scope["base_CAT1_and_SOD1"].sum())

        row = {
            "scope": scope_name,
            "n_total_rows": n_total,
            "n_base_CAT1_and_SOD1": n_base,
            "n_hit_6of6": int(df_scope["hit_6of6_under_base"].sum()),
            "n_hit_5of6": int(df_scope["hit_5of6_under_base"].sum()),
            "n_hit_4of6": int(df_scope["hit_4of6_under_base"].sum()),
            "n_hit_3of6": int(df_scope["hit_3of6_under_base"].sum()),
            "n_hit_2of6": int(df_scope["hit_2of6_under_base"].sum()),
        }

        for k in ["n_hit_6of6", "n_hit_5of6", "n_hit_4of6", "n_hit_3of6", "n_hit_2of6"]:
            row[f"{k}_rate_vs_total"] = (row[k] / n_total) if n_total > 0 else np.nan
            row[f"{k}_rate_vs_base"] = (row[k] / n_base) if n_base > 0 else np.nan

        return row

    rows = [summarize_scope(pred_all, "ALL")]

    if "source_dataset" in pred_all.columns:
        for ds in ["CAT", "SOD", "POD", "OXD"]:
            g = pred_all[pred_all["source_dataset"] == ds].copy()
            if len(g) > 0:
                rows.append(summarize_scope(g, ds))

    return pd.DataFrame(rows)


def print_summary(summary_df: pd.DataFrame):
    print("\n" + "=" * 110)
    print("Summary counts under base condition: CAT_any1==1 and SOD_any1==1")
    print("=" * 110)

    show_cols = [
        "scope",
        "n_total_rows",
        "n_base_CAT1_and_SOD1",
        "n_hit_6of6",
        "n_hit_5of6",
        "n_hit_4of6",
        "n_hit_3of6",
        "n_hit_2of6",
    ]
    print(summary_df[show_cols].to_string(index=False))
    print("=" * 110 + "\n")


# =========================================================
# 5) 主程序
# =========================================================
def main():
    # 读入并标准化列名
    df_cat = normalize_columns(pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME))
    df_sod = normalize_columns(pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME))
    df_pod = normalize_columns(pd.read_excel(POD_PATH, sheet_name=SHEET_NAME))
    df_oxd = normalize_columns(pd.read_excel(OXD_PATH, sheet_name=SHEET_NAME))

    # 标注来源
    for name, df in [("CAT", df_cat), ("SOD", df_sod), ("POD", df_pod), ("OXD", df_oxd)]:
        df["source_dataset"] = name
        df["orig_index"] = df.index.values

    # 补特征列
    df_cat = ensure_feature_columns(df_cat)
    df_sod = ensure_feature_columns(df_sod)
    df_pod = ensure_feature_columns(df_pod)
    df_oxd = ensure_feature_columns(df_oxd)

    # 检查关键列
    for col in [COL_KM, COL_VMAX, COL_KCAT]:
        if col not in df_cat.columns:
            raise ValueError(f"CAT 数据缺少列: {col}")

    for col in [COL_KM, COL_VMAX, COL_IC50]:
        if col not in df_sod.columns:
            raise ValueError(f"SOD 数据缺少列: {col}")

    # 合并预测集
    df_all = pd.concat([df_cat, df_sod, df_pod, df_oxd], ignore_index=True)

    # 训练 6 个模型
    cat_km_pipe, cat_v_pipe, cat_kc_pipe, cat_info = train_cat_models(df_cat)
    sod_km_pipe, sod_v_pipe, sod_ic_pipe, sod_info = train_sod_models(df_sod)

    # 保存模型
    joblib.dump(cat_km_pipe, OUT_DIR / "models" / "cat_km_low_gb.joblib")
    joblib.dump(cat_v_pipe,  OUT_DIR / "models" / "cat_vmax_high_dt.joblib")
    joblib.dump(cat_kc_pipe, OUT_DIR / "models" / "cat_kcat_high_knn.joblib")

    joblib.dump(sod_km_pipe, OUT_DIR / "models" / "sod_km_low_knn.joblib")
    joblib.dump(sod_v_pipe,  OUT_DIR / "models" / "sod_vmax_high_xgb.joblib")
    joblib.dump(sod_ic_pipe, OUT_DIR / "models" / "sod_ic50_low_xgb.joblib")

    # 全部样本预测
    pred_all = predict_all6(
        df_all,
        cat_km_pipe, cat_v_pipe, cat_kc_pipe,
        sod_km_pipe, sod_v_pipe, sod_ic_pipe
    )

    # 统计表
    summary_df = build_summary_table(pred_all)
    print_summary(summary_df)

    # 导出
    pred_all.to_csv(OUT_DIR / "all_predictions_all6.csv", index=False, encoding="utf-8-sig")
    summary_df.to_csv(OUT_DIR / "summary_counts_all6.csv", index=False, encoding="utf-8-sig")

    # 配置
    config = {
        "paths": {
            "CAT_PATH": CAT_PATH,
            "SOD_PATH": SOD_PATH,
            "POD_PATH": POD_PATH,
            "OXD_PATH": OXD_PATH,
        },
        "PROBA_THR": PROBA_THR,
        "CAT_fixed_thresholds_log10": {
            "KmLow_lo": CAT_KM_LO,
            "KmLow_hi": CAT_KM_HI,
            "VmaxHigh_thr": CAT_VMAX_THR,
            "KcatHigh_scheme": CAT_KCAT_SCHEME,
            "KcatHigh_q": CAT_KCAT_Q,
            "KcatHigh_thr_expected": CAT_KCAT_EXPECTED_THR,
        },
        "SOD_fixed_thresholds": {
            "IC50_uM": SOD_THR_IC50_UM,
            "IC50_ugmL": SOD_THR_IC50_UGML
        },
        "train_sanity_info": {
            **cat_info,
            **sod_info,
        },
        "prediction_rule": {
            "base_condition": "CAT三项至少1项为1 且 SOD三项至少1项为1",
            "count_bins": ["6/6", "5/6", "4/6", "3/6", "2/6"]
        },
        "exports": [
            "all_predictions_all6.csv",
            "summary_counts_all6.csv"
        ]
    }

    with open(OUT_DIR / "coarse_screen_all6_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    print("=" * 110)
    print("Done.")
    print(f"Total merged rows: {len(pred_all)}")
    print(f"Output dir: {OUT_DIR}")
    print("=" * 110)


if __name__ == "__main__":
    main()


Summary counts under base condition: CAT_any1==1 and SOD_any1==1
scope  n_total_rows  n_base_CAT1_and_SOD1  n_hit_6of6  n_hit_5of6  n_hit_4of6  n_hit_3of6  n_hit_2of6
  ALL           982                   929          65         268         305         219          72
  CAT            70                    56           2          13          15          18           8
  SOD            60                    46           0           9          15          15           7
  POD           611                   599          58         183         193         125          40
  OXD           241                   228           5          63          82          61          17

Done.
Total merged rows: 982
Output dir: C:\Users\86158\Desktop\coarse_screen_cat_sod_all6


In [5]:
# -*- coding: utf-8 -*-
"""
对 6 指标粗筛结果中的 5/6 和 6/6 候选，再用回归模型做二次排序

输入：
1) 前一步分类粗筛结果：
   all_predictions_all6.csv
   （要求至少包含：base_CAT1_and_SOD1, n_pos_ALL_6，以及原始特征列）
2) 训练回归模型用原始数据：
   - CAT: dataiscat.xlsx
   - SOD: dataissod.xlsx

回归模型（按你给的 best config，全量重训）：
A. SOD logKm
   model = SVR_RBF
   preprocess = {min_freq_cat:2, var_thr:0.0, k_best_req:None, svd_dim_req:8}
   model_params = {C:100, gamma:0.03, epsilon:0.03}

B. SOD log10(IC50, ug/mL)
   model = SVR_RBF
   preprocess = {min_freq_cat:3, var_thr:0.0, k_best_req:None, svd_dim_req:10}
   model_params = {C:30, gamma:0.01, epsilon:0.2}

C. CAT logKcat
   model = KernelRidge_RBF
   preprocess = {min_freq_cat:2, var_thr:0.0, k_best_req:15, svd_dim_req:6}
   model_params = {alpha:0.01, gamma:0.03}

排序逻辑：
- 先筛：base_CAT1_and_SOD1 == 1 且 n_pos_ALL_6 in [5, 6]
- 对候选样本预测：
    pred_logKm_SOD
    pred_logIC50_ugmL_SOD
    pred_logKcat_CAT
- 再转成 0~1 的分位分数：
    score_SOD_Km_pct            (Km 越小越好)
    score_SOD_IC50_ugmL_pct    (IC50 越小越好)
    score_CAT_Kcat_pct         (Kcat 越大越好)
- 加权：
    score_SOD_avg_0_1 = (score_SOD_Km_pct + score_SOD_IC50_ugmL_pct) / 2
    score_final_50_50 = 0.5 * score_CAT_Kcat_pct + 0.5 * score_SOD_avg_0_1

输出：
OUT_DIR/
  - ranked_6of6_by_regression.csv
  - ranked_5of6_by_regression.csv
  - ranked_5or6_combined_by_regression.csv
  - regression_ranking_config.json
  - models/
      sod_logkm_svr_rbf_refit_full.joblib
      sod_ic50_ugml_svr_rbf_refit_full.joblib
      cat_logkcat_krr_rbf_refit_full.joblib
"""

import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold, f_regression
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge


# =========================================================
# 0) 路径
# =========================================================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"

# 前一步 6 指标粗筛结果表
SCREEN_TABLE_PATH = r"C:\Users\86158\Desktop\coarse_screen_cat_sod_all6\all_predictions_all6.csv"

OUT_DIR = Path(r"C:\Users\86158\Desktop\coarse_screen_cat_sod_all6_reg_rank")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)

SHEET_NAME = 0
RANDOM_SEED = 42

COL_DOI  = "data reference doi"
COL_KM   = "Km/mM"
COL_KCAT = "Kcat/s-1"
COL_IC50 = "IC50/μM"


# =========================================================
# 1) 你训练时的固定最佳参数
# =========================================================
SOD_LOGKM_CFG = {
    "model_name": "SVR_RBF",
    "min_freq_cat": 2,
    "var_thr": 0.0,
    "k_best_req": None,
    "svd_dim_req": 8,
    "model_params": {"C": 100, "gamma": 0.03, "epsilon": 0.03},
}

SOD_IC50_UGML_CFG = {
    "model_name": "SVR_RBF",
    "min_freq_cat": 3,
    "var_thr": 0.0,
    "k_best_req": None,
    "svd_dim_req": 10,
    "model_params": {"C": 30, "gamma": 0.01, "epsilon": 0.2},
}

CAT_LOGKCAT_CFG = {
    "model_name": "KernelRidge_RBF",
    "min_freq_cat": 2,
    "var_thr": 0.0,
    "k_best_req": 15,
    "svd_dim_req": 6,
    "model_params": {"alpha": 0.01, "gamma": 0.03},
}


# =========================================================
# 2) 特征列（按 normalize_columns 之后的规范名）
# =========================================================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate"
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = [
    "shape", "surface treatment", "dispersion medium", "Substrate"
]


# =========================================================
# 3) 基础工具
# =========================================================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def ensure_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in FEATURE_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df


def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    df_raw = ensure_feature_columns(df_raw)
    X = df_raw[FEATURE_COLS].copy()

    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0.0)

    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X


def parse_ic50_cell(x):
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")
    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"
    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


# =========================================================
# 4) 复刻你原回归代码里的预处理模块
# =========================================================
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """freq < min_freq -> __other__"""
    def __init__(self, min_freq=3, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


class SelectKBestSafe(BaseEstimator, TransformerMixin):
    def __init__(self, k=None):
        self.k = k
        self.selector_ = None

    def fit(self, X, y):
        if self.k is None:
            self.selector_ = None
            return self

        n_feat = X.shape[1]
        k_eff = min(int(self.k), int(n_feat))
        if k_eff <= 0:
            self.selector_ = None
            return self

        from sklearn.feature_selection import SelectKBest
        self.selector_ = SelectKBest(score_func=f_regression, k=k_eff)
        self.selector_.fit(X, y)
        return self

    def transform(self, X):
        if self.selector_ is None:
            return X
        return self.selector_.transform(X)


class TruncatedSVDSafe(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=None, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.svd_ = None

    def fit(self, X, y=None):
        if self.n_components is None:
            self.svd_ = None
            return self

        n_feat = X.shape[1]
        if n_feat <= 1:
            self.svd_ = None
            return self

        n_eff = min(int(self.n_components), max(1, n_feat - 1))
        self.svd_ = TruncatedSVD(n_components=n_eff, random_state=self.random_state)
        self.svd_.fit(X)
        return self

    def transform(self, X):
        if self.svd_ is None:
            return X
        return self.svd_.transform(X)


def build_preprocess(min_freq_cat=3, sparse=True):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=sparse)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=min_freq_cat)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop",
    )


def instantiate_regressor(model_name, params):
    if model_name == "SVR_RBF":
        return SVR(kernel="rbf", **params)
    if model_name == "KernelRidge_RBF":
        return KernelRidge(kernel="rbf", **params)
    raise ValueError(model_name)


def build_fixed_reg_pipe(cfg: dict):
    steps = []
    steps.append(("preprocess", build_preprocess(min_freq_cat=cfg["min_freq_cat"], sparse=True)))
    steps.append(("var", VarianceThreshold(threshold=float(cfg["var_thr"]))))

    if cfg["k_best_req"] is not None:
        steps.append(("kbest", SelectKBestSafe(k=int(cfg["k_best_req"]))))

    if cfg["svd_dim_req"] is not None:
        steps.append(("svd", TruncatedSVDSafe(
            n_components=int(cfg["svd_dim_req"]),
            random_state=RANDOM_SEED
        )))

    # 你给的 3 个 best config 都带了 SVD，所以这里统一接 dense scaler
    steps.append(("scaler", StandardScaler(with_mean=True)))
    steps.append(("model", instantiate_regressor(cfg["model_name"], cfg["model_params"])))

    return Pipeline(steps)


# =========================================================
# 5) 三个回归模型的全量重训
# =========================================================
def train_sod_logkm_model(df_sod: pd.DataFrame):
    v = pd.to_numeric(df_sod[COL_KM], errors="coerce")
    keep = v.notna() & (v > 0)
    df_use = df_sod.loc[keep].copy()

    X = prepare_features(df_use)
    y = np.log10(pd.to_numeric(df_use[COL_KM], errors="coerce").astype(float).values)

    pipe = build_fixed_reg_pipe(SOD_LOGKM_CFG)
    pipe.fit(X, y)

    return pipe, {
        "n_train": int(len(df_use)),
        "target": "log10(Km/mM)",
        **SOD_LOGKM_CFG
    }


def train_sod_ic50_ugml_model(df_sod: pd.DataFrame):
    df2 = df_sod.copy()
    parsed = df2[COL_IC50].apply(parse_ic50_cell)
    df2["IC50_value"] = parsed.apply(lambda t: t[0])
    df2["IC50_unit"]  = parsed.apply(lambda t: t[1])

    keep = (
        (df2["IC50_unit"] == "ug/mL") &
        df2["IC50_value"].notna() &
        (pd.to_numeric(df2["IC50_value"], errors="coerce") > 0)
    )
    df_use = df2.loc[keep].copy()

    X = prepare_features(df_use)
    y = np.log10(pd.to_numeric(df_use["IC50_value"], errors="coerce").astype(float).values)

    pipe = build_fixed_reg_pipe(SOD_IC50_UGML_CFG)
    pipe.fit(X, y)

    return pipe, {
        "n_train": int(len(df_use)),
        "target": "log10(IC50 ug/mL)",
        **SOD_IC50_UGML_CFG
    }


def train_cat_logkcat_model(df_cat: pd.DataFrame):
    v = pd.to_numeric(df_cat[COL_KCAT], errors="coerce")
    keep = v.notna() & (v > 0)
    df_use = df_cat.loc[keep].copy()

    X = prepare_features(df_use)
    y = np.log10(pd.to_numeric(df_use[COL_KCAT], errors="coerce").astype(float).values)

    pipe = build_fixed_reg_pipe(CAT_LOGKCAT_CFG)
    pipe.fit(X, y)

    return pipe, {
        "n_train": int(len(df_use)),
        "target": "log10(Kcat/s-1)",
        **CAT_LOGKCAT_CFG
    }


# =========================================================
# 6) 排序分数
# =========================================================
def percentile_score(series, higher_is_better=True):
    """
    转成 0~1 的分位分数，便于不同回归模型加权。
    """
    s = pd.Series(series, copy=True).astype(float)
    desirability = s if higher_is_better else (-s)
    return desirability.rank(method="average", pct=True)


def add_regression_ranking_columns(cand: pd.DataFrame) -> pd.DataFrame:
    out = cand.copy()

    # 原始 log 预测
    out["pred_logKm_SOD_reg"] = out["pred_logKm_SOD_reg"].astype(float)
    out["pred_logIC50_ugmL_SOD_reg"] = out["pred_logIC50_ugmL_SOD_reg"].astype(float)
    out["pred_logKcat_CAT_reg"] = out["pred_logKcat_CAT_reg"].astype(float)

    # 还原到原单位，方便人工看
    out["pred_Km_mM_SOD_reg"] = np.power(10.0, out["pred_logKm_SOD_reg"])
    out["pred_IC50_ugmL_SOD_reg"] = np.power(10.0, out["pred_logIC50_ugmL_SOD_reg"])
    out["pred_Kcat_s-1_CAT_reg"] = np.power(10.0, out["pred_logKcat_CAT_reg"])

    # 单项分位得分
    out["score_SOD_Km_pct"] = percentile_score(out["pred_logKm_SOD_reg"], higher_is_better=False)
    out["score_SOD_IC50_ugmL_pct"] = percentile_score(out["pred_logIC50_ugmL_SOD_reg"], higher_is_better=False)
    out["score_CAT_Kcat_pct"] = percentile_score(out["pred_logKcat_CAT_reg"], higher_is_better=True)

    # 组内得分
    out["score_SOD_avg_0_1"] = (
        out["score_SOD_Km_pct"] + out["score_SOD_IC50_ugmL_pct"]
    ) / 2.0

    # 最终：SOD 50%，CAT 50%
    out["score_final_50_50"] = (
        0.5 * out["score_CAT_Kcat_pct"] +
        0.5 * out["score_SOD_avg_0_1"]
    )

    return out


# =========================================================
# 7) 主程序
# =========================================================
def main():
    # ---------- 读原始训练数据 ----------
    df_cat = normalize_columns(pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME))
    df_sod = normalize_columns(pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME))

    # ---------- 全量重训 3 个回归器 ----------
    sod_km_pipe, sod_km_info = train_sod_logkm_model(df_sod)
    sod_ic50_pipe, sod_ic50_info = train_sod_ic50_ugml_model(df_sod)
    cat_kcat_pipe, cat_kcat_info = train_cat_logkcat_model(df_cat)

    joblib.dump(sod_km_pipe, OUT_DIR / "models" / "sod_logkm_svr_rbf_refit_full.joblib")
    joblib.dump(sod_ic50_pipe, OUT_DIR / "models" / "sod_ic50_ugml_svr_rbf_refit_full.joblib")
    joblib.dump(cat_kcat_pipe, OUT_DIR / "models" / "cat_logkcat_krr_rbf_refit_full.joblib")

    # ---------- 读 6 指标粗筛结果 ----------
    screen_df = pd.read_csv(SCREEN_TABLE_PATH, encoding="utf-8-sig")
    screen_df = normalize_columns(screen_df)

    need_cols = ["base_CAT1_and_SOD1", "n_pos_ALL_6"]
    miss = [c for c in need_cols if c not in screen_df.columns]
    if miss:
        raise ValueError(f"粗筛结果表缺少关键列: {miss}")

    # 只取 5/6 和 6/6
    cand = screen_df[
        (screen_df["base_CAT1_and_SOD1"] == 1) &
        (screen_df["n_pos_ALL_6"].isin([5, 6]))
    ].copy()

    if cand.empty:
        raise RuntimeError("在 all_predictions_all6.csv 中没有找到满足 base 条件且属于 5/6 或 6/6 的候选。")

    # ---------- 对候选样本做回归预测 ----------
    X_cand = prepare_features(cand)

    cand["pred_logKm_SOD_reg"] = sod_km_pipe.predict(X_cand)
    cand["pred_logIC50_ugmL_SOD_reg"] = sod_ic50_pipe.predict(X_cand)
    cand["pred_logKcat_CAT_reg"] = cat_kcat_pipe.predict(X_cand)

    # ---------- 计算排序分数 ----------
    cand = add_regression_ranking_columns(cand)

    # 组内排序
    cand_6 = cand[cand["n_pos_ALL_6"] == 6].copy().sort_values(
        ["score_final_50_50", "score_CAT_Kcat_pct", "score_SOD_avg_0_1"],
        ascending=[False, False, False]
    )
    cand_5 = cand[cand["n_pos_ALL_6"] == 5].copy().sort_values(
        ["score_final_50_50", "score_CAT_Kcat_pct", "score_SOD_avg_0_1"],
        ascending=[False, False, False]
    )

    cand_6["rank_in_6of6"] = np.arange(1, len(cand_6) + 1)
    cand_5["rank_in_5of6"] = np.arange(1, len(cand_5) + 1)

    # 合并总表：先 6/6 后 5/6，组内按回归分数排
    cand_all = pd.concat([cand_6, cand_5], axis=0, ignore_index=True)
    cand_all["rank_overall_bucket_then_reg"] = np.arange(1, len(cand_all) + 1)

    # ---------- 导出 ----------
    cand_6.to_csv(OUT_DIR / "ranked_6of6_by_regression.csv", index=False, encoding="utf-8-sig")
    cand_5.to_csv(OUT_DIR / "ranked_5of6_by_regression.csv", index=False, encoding="utf-8-sig")
    cand_all.to_csv(OUT_DIR / "ranked_5or6_combined_by_regression.csv", index=False, encoding="utf-8-sig")

    config = {
        "input": {
            "CAT_PATH": CAT_PATH,
            "SOD_PATH": SOD_PATH,
            "SCREEN_TABLE_PATH": SCREEN_TABLE_PATH,
        },
        "candidate_filter": {
            "base_condition": "base_CAT1_and_SOD1 == 1",
            "n_pos_ALL_6_in": [5, 6],
        },
        "regression_models_refit_full": {
            "SOD_logKm": sod_km_info,
            "SOD_logIC50_ugmL": sod_ic50_info,
            "CAT_logKcat": cat_kcat_info,
        },
        "ranking_logic": {
            "score_direction": {
                "SOD_logKm": "lower is better",
                "SOD_logIC50_ugmL": "lower is better",
                "CAT_logKcat": "higher is better",
            },
            "normalize_method": "percentile rank within the 5/6 + 6/6 candidate pool",
            "weights": {
                "CAT_total": 0.5,
                "SOD_total": 0.5,
                "SOD_logKm_within_SOD": 0.5,
                "SOD_logIC50_ugmL_within_SOD": 0.5,
            },
            "formula": "score_final_50_50 = 0.5 * score_CAT_Kcat_pct + 0.5 * ((score_SOD_Km_pct + score_SOD_IC50_ugmL_pct) / 2)"
        },
        "exports": [
            "ranked_6of6_by_regression.csv",
            "ranked_5of6_by_regression.csv",
            "ranked_5or6_combined_by_regression.csv",
        ]
    }

    with open(OUT_DIR / "regression_ranking_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    print("=" * 96)
    print("Done.")
    print(f"5/6 candidates: {len(cand_5)}")
    print(f"6/6 candidates: {len(cand_6)}")
    print("Output dir:", str(OUT_DIR))
    print("=" * 96)


if __name__ == "__main__":
    main()

Done.
5/6 candidates: 268
6/6 candidates: 65
Output dir: C:\Users\86158\Desktop\coarse_screen_cat_sod_all6_reg_rank


In [6]:
# -*- coding: utf-8 -*-
"""
四类 relaxed(5/6 or 6/6 match target pattern) 分型 + 回归二次排序

分型规则：
1) highCAT_highSOD: 目标模式 111111，匹配数 >= 5
2) highCAT_lowSOD : 目标模式 111000，匹配数 >= 5
3) lowCAT_highSOD : 目标模式 000111，匹配数 >= 5
4) lowCAT_lowSOD  : 目标模式 000000，匹配数 >= 5

其中 6 位顺序固定为：
[pred_CAT_KmLow, pred_CAT_VmaxHigh, pred_CAT_KcatHigh,
 pred_SOD_KmLow, pred_SOD_VmaxHigh, pred_SOD_IC50Low]

回归模型（全量重训）：
- SOD logKm          -> SVR_RBF
- SOD logIC50_ugmL   -> SVR_RBF
- CAT logKcat        -> KernelRidge_RBF

排序逻辑（组内）：
- CAT 50%
- SOD 50%
- SOD内部：Km 和 IC50 各占 50%

方向：
- 高CAT：Kcat 越大越好
- 低CAT：Kcat 越小越好
- 高SOD：Km 越小越好，IC50 越小越好
- 低SOD：Km 越大越好，IC50 越大越好

输出：
OUT_DIR/
  - all_with_relaxed_groups_and_regression_scores.csv
  - rank_highCAT_highSOD_relaxed_5of6.csv
  - rank_highCAT_lowSOD_relaxed_5of6.csv
  - rank_lowCAT_highSOD_relaxed_5of6.csv
  - rank_lowCAT_lowSOD_relaxed_5of6.csv
  - relaxed_4groups_summary.csv
  - relaxed_4groups_config.json
  - models/
      sod_logkm_svr_rbf_refit_full.joblib
      sod_ic50_ugml_svr_rbf_refit_full.joblib
      cat_logkcat_krr_rbf_refit_full.joblib
"""

import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold, f_regression
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge


# =========================================================
# 0) 路径
# =========================================================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"
SOD_PATH = r"C:\Users\86158\Desktop\dataissod.xlsx"
SCREEN_TABLE_PATH = r"C:\Users\86158\Desktop\coarse_screen_cat_sod_all6\all_predictions_all6.csv"

OUT_DIR = Path(r"C:\Users\86158\Desktop\relaxed_4groups_regression_rank")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)

SHEET_NAME = 0
RANDOM_SEED = 42

COL_DOI  = "data reference doi"
COL_KM   = "Km/mM"
COL_KCAT = "Kcat/s-1"
COL_IC50 = "IC50/μM"


# =========================================================
# 1) 固定 best 参数（按你给的结果）
# =========================================================
SOD_LOGKM_CFG = {
    "model_name": "SVR_RBF",
    "min_freq_cat": 2,
    "var_thr": 0.0,
    "k_best_req": None,
    "svd_dim_req": 8,
    "model_params": {"C": 100, "gamma": 0.03, "epsilon": 0.03},
}

SOD_IC50_UGML_CFG = {
    "model_name": "SVR_RBF",
    "min_freq_cat": 3,
    "var_thr": 0.0,
    "k_best_req": None,
    "svd_dim_req": 10,
    "model_params": {"C": 30, "gamma": 0.01, "epsilon": 0.2},
}

CAT_LOGKCAT_CFG = {
    "model_name": "KernelRidge_RBF",
    "min_freq_cat": 2,
    "var_thr": 0.0,
    "k_best_req": 15,
    "svd_dim_req": 6,
    "model_params": {"alpha": 0.01, "gamma": 0.03},
}


# =========================================================
# 2) 特征列
# =========================================================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate"
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = [
    "shape", "surface treatment", "dispersion medium", "Substrate"
]


# =========================================================
# 3) 基础工具
# =========================================================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def ensure_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in FEATURE_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df


def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    df_raw = ensure_feature_columns(df_raw)
    X = df_raw[FEATURE_COLS].copy()

    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0.0)

    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X


def parse_ic50_cell(x):
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")
    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"
    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


# =========================================================
# 4) 预处理模块（按你回归训练代码复刻）
# =========================================================
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_freq=3, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


class SelectKBestSafe(BaseEstimator, TransformerMixin):
    def __init__(self, k=None):
        self.k = k
        self.selector_ = None

    def fit(self, X, y):
        if self.k is None:
            self.selector_ = None
            return self

        n_feat = X.shape[1]
        k_eff = min(int(self.k), int(n_feat))
        if k_eff <= 0:
            self.selector_ = None
            return self

        from sklearn.feature_selection import SelectKBest
        self.selector_ = SelectKBest(score_func=f_regression, k=k_eff)
        self.selector_.fit(X, y)
        return self

    def transform(self, X):
        if self.selector_ is None:
            return X
        return self.selector_.transform(X)


class TruncatedSVDSafe(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=None, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.svd_ = None

    def fit(self, X, y=None):
        if self.n_components is None:
            self.svd_ = None
            return self

        n_feat = X.shape[1]
        if n_feat <= 1:
            self.svd_ = None
            return self

        n_eff = min(int(self.n_components), max(1, n_feat - 1))
        self.svd_ = TruncatedSVD(n_components=n_eff, random_state=self.random_state)
        self.svd_.fit(X)
        return self

    def transform(self, X):
        if self.svd_ is None:
            return X
        return self.svd_.transform(X)


def build_preprocess(min_freq_cat=3, sparse=True):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=sparse)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=min_freq_cat)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop",
    )


def instantiate_regressor(model_name, params):
    if model_name == "SVR_RBF":
        return SVR(kernel="rbf", **params)
    if model_name == "KernelRidge_RBF":
        return KernelRidge(kernel="rbf", **params)
    raise ValueError(model_name)


def build_fixed_reg_pipe(cfg: dict):
    steps = []
    steps.append(("preprocess", build_preprocess(min_freq_cat=cfg["min_freq_cat"], sparse=True)))
    steps.append(("var", VarianceThreshold(threshold=float(cfg["var_thr"]))))

    if cfg["k_best_req"] is not None:
        steps.append(("kbest", SelectKBestSafe(k=int(cfg["k_best_req"]))))

    if cfg["svd_dim_req"] is not None:
        steps.append((
            "svd",
            TruncatedSVDSafe(
                n_components=int(cfg["svd_dim_req"]),
                random_state=RANDOM_SEED
            )
        ))

    steps.append(("scaler", StandardScaler(with_mean=True)))
    steps.append(("model", instantiate_regressor(cfg["model_name"], cfg["model_params"])))
    return Pipeline(steps)


# =========================================================
# 5) 全量重训三个回归模型
# =========================================================
def train_sod_logkm_model(df_sod: pd.DataFrame):
    v = pd.to_numeric(df_sod[COL_KM], errors="coerce")
    keep = v.notna() & (v > 0)
    df_use = df_sod.loc[keep].copy()

    if df_use.empty:
        raise RuntimeError("SOD 中没有可用于训练 logKm 回归的有效样本。")

    X = prepare_features(df_use)
    y = np.log10(pd.to_numeric(df_use[COL_KM], errors="coerce").astype(float).values)

    pipe = build_fixed_reg_pipe(SOD_LOGKM_CFG)
    pipe.fit(X, y)
    return pipe, {"n_train": int(len(df_use)), "target": "log10(Km/mM)", **SOD_LOGKM_CFG}


def train_sod_ic50_ugml_model(df_sod: pd.DataFrame):
    df2 = df_sod.copy()
    parsed = df2[COL_IC50].apply(parse_ic50_cell)
    df2["IC50_value"] = parsed.apply(lambda t: t[0])
    df2["IC50_unit"] = parsed.apply(lambda t: t[1])

    keep = (
        (df2["IC50_unit"] == "ug/mL") &
        df2["IC50_value"].notna() &
        (pd.to_numeric(df2["IC50_value"], errors="coerce") > 0)
    )
    df_use = df2.loc[keep].copy()

    if df_use.empty:
        raise RuntimeError("SOD 中没有可用于训练 logIC50(ug/mL) 回归的有效样本。")

    X = prepare_features(df_use)
    y = np.log10(pd.to_numeric(df_use["IC50_value"], errors="coerce").astype(float).values)

    pipe = build_fixed_reg_pipe(SOD_IC50_UGML_CFG)
    pipe.fit(X, y)
    return pipe, {"n_train": int(len(df_use)), "target": "log10(IC50 ug/mL)", **SOD_IC50_UGML_CFG}


def train_cat_logkcat_model(df_cat: pd.DataFrame):
    v = pd.to_numeric(df_cat[COL_KCAT], errors="coerce")
    keep = v.notna() & (v > 0)
    df_use = df_cat.loc[keep].copy()

    if df_use.empty:
        raise RuntimeError("CAT 中没有可用于训练 logKcat 回归的有效样本。")

    X = prepare_features(df_use)
    y = np.log10(pd.to_numeric(df_use[COL_KCAT], errors="coerce").astype(float).values)

    pipe = build_fixed_reg_pipe(CAT_LOGKCAT_CFG)
    pipe.fit(X, y)
    return pipe, {"n_train": int(len(df_use)), "target": "log10(Kcat/s-1)", **CAT_LOGKCAT_CFG}


# =========================================================
# 6) relaxed 分组：匹配目标模式 >= 5/6
# =========================================================
CAT_COLS = ["pred_CAT_KmLow", "pred_CAT_VmaxHigh", "pred_CAT_KcatHigh"]
SOD_COLS = ["pred_SOD_KmLow", "pred_SOD_VmaxHigh", "pred_SOD_IC50Low"]
ALL6_COLS = CAT_COLS + SOD_COLS


def add_relaxed_group_flags(df: pd.DataFrame, min_match=5) -> pd.DataFrame:
    """
    四组 relaxed 定义：
    - highCAT_highSOD : 目标 111111，匹配数 >= 5
    - highCAT_lowSOD  : 目标 111000，匹配数 >= 5
    - lowCAT_highSOD  : 目标 000111，匹配数 >= 5
    - lowCAT_lowSOD   : 目标 000000，匹配数 >= 5
    """
    out = df.copy()

    missing = [c for c in ALL6_COLS if c not in out.columns]
    if missing:
        raise ValueError(f"粗筛表缺少以下分类预测列: {missing}")

    pred = out[ALL6_COLS].astype(int).values

    target_hh = np.array([1, 1, 1, 1, 1, 1], dtype=int)
    target_hl = np.array([1, 1, 1, 0, 0, 0], dtype=int)
    target_lh = np.array([0, 0, 0, 1, 1, 1], dtype=int)
    target_ll = np.array([0, 0, 0, 0, 0, 0], dtype=int)

    out["match_n_highCAT_highSOD"] = (pred == target_hh).sum(axis=1)
    out["match_n_highCAT_lowSOD"] = (pred == target_hl).sum(axis=1)
    out["match_n_lowCAT_highSOD"] = (pred == target_lh).sum(axis=1)
    out["match_n_lowCAT_lowSOD"] = (pred == target_ll).sum(axis=1)

    out["group_highCAT_highSOD"] = (out["match_n_highCAT_highSOD"] >= min_match).astype(int)
    out["group_highCAT_lowSOD"] = (out["match_n_highCAT_lowSOD"] >= min_match).astype(int)
    out["group_lowCAT_highSOD"] = (out["match_n_lowCAT_highSOD"] >= min_match).astype(int)
    out["group_lowCAT_lowSOD"] = (out["match_n_lowCAT_lowSOD"] >= min_match).astype(int)

    return out


# =========================================================
# 7) 回归预测 + 排序
# =========================================================
def percentile_score(series, higher_is_better=True):
    s = pd.Series(series, copy=True).astype(float)
    desirability = s if higher_is_better else (-s)
    return desirability.rank(method="average", pct=True)


def add_regression_predictions(df: pd.DataFrame, sod_km_pipe, sod_ic50_pipe, cat_kcat_pipe) -> pd.DataFrame:
    out = df.copy()
    X = prepare_features(out)

    out["pred_logKm_SOD_reg"] = sod_km_pipe.predict(X)
    out["pred_logIC50_ugmL_SOD_reg"] = sod_ic50_pipe.predict(X)
    out["pred_logKcat_CAT_reg"] = cat_kcat_pipe.predict(X)

    out["pred_Km_mM_SOD_reg"] = np.power(10.0, out["pred_logKm_SOD_reg"])
    out["pred_IC50_ugmL_SOD_reg"] = np.power(10.0, out["pred_logIC50_ugmL_SOD_reg"])
    out["pred_Kcat_s-1_CAT_reg"] = np.power(10.0, out["pred_logKcat_CAT_reg"])

    return out


def rank_one_group(df_group: pd.DataFrame, cat_high: bool, sod_high: bool, group_name: str) -> pd.DataFrame:
    out = df_group.copy()
    if out.empty:
        return out

    # CAT: 高 -> Kcat大好；低 -> Kcat小好
    out["score_CAT_component_0_1"] = percentile_score(
        out["pred_logKcat_CAT_reg"],
        higher_is_better=cat_high
    )

    # SOD: 高 -> Km/IC50小好；低 -> Km/IC50大好
    out["score_SOD_Km_component_0_1"] = percentile_score(
        out["pred_logKm_SOD_reg"],
        higher_is_better=(not sod_high)
    )
    out["score_SOD_IC50_component_0_1"] = percentile_score(
        out["pred_logIC50_ugmL_SOD_reg"],
        higher_is_better=(not sod_high)
    )

    out["score_SOD_component_0_1"] = (
        out["score_SOD_Km_component_0_1"] +
        out["score_SOD_IC50_component_0_1"]
    ) / 2.0

    out["score_final_50_50"] = (
        0.5 * out["score_CAT_component_0_1"] +
        0.5 * out["score_SOD_component_0_1"]
    )

    out = out.sort_values(
        ["score_final_50_50", "score_CAT_component_0_1", "score_SOD_component_0_1"],
        ascending=[False, False, False]
    ).copy()

    out["rank_in_group"] = np.arange(1, len(out) + 1)
    out["relaxed_group_name"] = group_name

    return out


# =========================================================
# 8) 主程序
# =========================================================
def main():
    # ---------- 读原始训练数据 ----------
    df_cat = normalize_columns(pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME))
    df_sod = normalize_columns(pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME))

    # ---------- 全量重训 3 个回归器 ----------
    sod_km_pipe, sod_km_info = train_sod_logkm_model(df_sod)
    sod_ic50_pipe, sod_ic50_info = train_sod_ic50_ugml_model(df_sod)
    cat_kcat_pipe, cat_kcat_info = train_cat_logkcat_model(df_cat)

    joblib.dump(sod_km_pipe, OUT_DIR / "models" / "sod_logkm_svr_rbf_refit_full.joblib")
    joblib.dump(sod_ic50_pipe, OUT_DIR / "models" / "sod_ic50_ugml_svr_rbf_refit_full.joblib")
    joblib.dump(cat_kcat_pipe, OUT_DIR / "models" / "cat_logkcat_krr_rbf_refit_full.joblib")

    # ---------- 读 6 指标粗筛结果 ----------
    screen_df = pd.read_csv(SCREEN_TABLE_PATH, encoding="utf-8-sig")
    screen_df = normalize_columns(screen_df)

    # ---------- 加 relaxed 分组 ----------
    screen_df = add_relaxed_group_flags(screen_df, min_match=5)

    # ---------- 加回归预测 ----------
    screen_df = add_regression_predictions(screen_df, sod_km_pipe, sod_ic50_pipe, cat_kcat_pipe)

    # 保存全表（保留原表全部列 + 新增列）
    screen_df.to_csv(
        OUT_DIR / "all_with_relaxed_groups_and_regression_scores.csv",
        index=False, encoding="utf-8-sig"
    )

    # ---------- 四组分别排序 ----------
    g_hh = rank_one_group(
        screen_df[screen_df["group_highCAT_highSOD"] == 1].copy(),
        cat_high=True, sod_high=True,
        group_name="highCAT_highSOD"
    )

    g_hl = rank_one_group(
        screen_df[screen_df["group_highCAT_lowSOD"] == 1].copy(),
        cat_high=True, sod_high=False,
        group_name="highCAT_lowSOD"
    )

    g_lh = rank_one_group(
        screen_df[screen_df["group_lowCAT_highSOD"] == 1].copy(),
        cat_high=False, sod_high=True,
        group_name="lowCAT_highSOD"
    )

    g_ll = rank_one_group(
        screen_df[screen_df["group_lowCAT_lowSOD"] == 1].copy(),
        cat_high=False, sod_high=False,
        group_name="lowCAT_lowSOD"
    )

    # ---------- 导出 ----------
    g_hh.to_csv(OUT_DIR / "rank_highCAT_highSOD_relaxed_5of6.csv", index=False, encoding="utf-8-sig")
    g_hl.to_csv(OUT_DIR / "rank_highCAT_lowSOD_relaxed_5of6.csv", index=False, encoding="utf-8-sig")
    g_lh.to_csv(OUT_DIR / "rank_lowCAT_highSOD_relaxed_5of6.csv", index=False, encoding="utf-8-sig")
    g_ll.to_csv(OUT_DIR / "rank_lowCAT_lowSOD_relaxed_5of6.csv", index=False, encoding="utf-8-sig")

    summary = pd.DataFrame([
        {
            "group": "highCAT_highSOD",
            "n": len(g_hh),
            "match_rule": ">=5 of target 111111"
        },
        {
            "group": "highCAT_lowSOD",
            "n": len(g_hl),
            "match_rule": ">=5 of target 111000"
        },
        {
            "group": "lowCAT_highSOD",
            "n": len(g_lh),
            "match_rule": ">=5 of target 000111"
        },
        {
            "group": "lowCAT_lowSOD",
            "n": len(g_ll),
            "match_rule": ">=5 of target 000000"
        },
    ])
    summary.to_csv(OUT_DIR / "relaxed_4groups_summary.csv", index=False, encoding="utf-8-sig")

    config = {
        "input": {
            "CAT_PATH": CAT_PATH,
            "SOD_PATH": SOD_PATH,
            "SCREEN_TABLE_PATH": SCREEN_TABLE_PATH,
        },
        "relaxed_definition": {
            "highCAT_highSOD": {
                "target_pattern": [1, 1, 1, 1, 1, 1],
                "min_match": 5
            },
            "highCAT_lowSOD": {
                "target_pattern": [1, 1, 1, 0, 0, 0],
                "min_match": 5
            },
            "lowCAT_highSOD": {
                "target_pattern": [0, 0, 0, 1, 1, 1],
                "min_match": 5
            },
            "lowCAT_lowSOD": {
                "target_pattern": [0, 0, 0, 0, 0, 0],
                "min_match": 5
            }
        },
        "regression_models_refit_full": {
            "SOD_logKm": sod_km_info,
            "SOD_logIC50_ugmL": sod_ic50_info,
            "CAT_logKcat": cat_kcat_info,
        },
        "ranking_logic": {
            "CAT_weight": 0.5,
            "SOD_weight": 0.5,
            "SOD_Km_within_SOD": 0.5,
            "SOD_IC50_within_SOD": 0.5,
            "direction": {
                "highCAT": "Kcat higher is better",
                "lowCAT": "Kcat lower is better",
                "highSOD": "Km lower + IC50 lower are better",
                "lowSOD": "Km higher + IC50 higher are better"
            }
        },
        "exports": [
            "all_with_relaxed_groups_and_regression_scores.csv",
            "rank_highCAT_highSOD_relaxed_5of6.csv",
            "rank_highCAT_lowSOD_relaxed_5of6.csv",
            "rank_lowCAT_highSOD_relaxed_5of6.csv",
            "rank_lowCAT_lowSOD_relaxed_5of6.csv",
            "relaxed_4groups_summary.csv"
        ]
    }

    with open(OUT_DIR / "relaxed_4groups_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    print("=" * 100)
    print("Done.")
    print(summary.to_string(index=False))
    print("Outputs:", str(OUT_DIR))
    print("=" * 100)


if __name__ == "__main__":
    main()

Done.
          group   n           match_rule
highCAT_highSOD 333 >=5 of target 111111
 highCAT_lowSOD  90 >=5 of target 111000
 lowCAT_highSOD  58 >=5 of target 000111
  lowCAT_lowSOD  22 >=5 of target 000000
Outputs: C:\Users\86158\Desktop\relaxed_4groups_regression_rank


In [7]:
# -*- coding: utf-8 -*-
"""
六个分类模型（CAT/SOD）统一原始特征 SHAP 分析 —— 稳定汇总版
========================================================
修复重点：
1) 不把 object/string 直接传给 shap
2) SHAP 使用纯数值编码矩阵
3) 预测时把类别 code 反解成原始类别字符串，再送入 pipeline
4) 不使用 shap.summary_plot，改为自己画图，避免 isfinite 兼容问题
5) 自动输出“全模型汇总大表”

输出：
A. 每个模型子目录：
- model_info.json
- sample_level_shap.csv
- global_importance.csv
- numeric_feature_summary.csv
- numeric_feature_bins.csv
- categorical_feature_levels.csv
- global_importance_bar.png
- top_features_scatter.png

B. 总汇总：
- all_models_summary.xlsx
- combined_global_importance.csv
- combined_numeric_feature_summary.csv
- combined_numeric_feature_bins.csv
- combined_categorical_feature_levels.csv
- combined_model_info.csv

解释规则：
- 所有任务都按 class 1 = 好活性
- 因此 SHAP > 0 表示该特征把样本推向“好活性”
"""

import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import shap

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


# =========================================================
# 0) 路径与参数
# =========================================================
CAT_PATH = Path(r"C:\Users\86158\Desktop\dataiscat.xlsx")
SOD_PATH = Path(r"C:\Users\86158\Desktop\dataissod.xlsx")
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\all6_classifier_raw_shap_summary")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

# SHAP 参数
N_BACKGROUND = 30
MAX_EVALS_FACTOR = 8
N_BINS_NUMERIC = 5
TOPN_PLOT = 15


# =========================================================
# 1) 常量
# =========================================================
COL_KM   = "Km/mM"
COL_VMAX = "Vmax/μM s-1"
COL_KCAT = "Kcat/s-1"
COL_IC50 = "IC50/μM"

CAT_KM_LO    = 1.0253058652647702
CAT_KM_HI    = 1.8055008581584002
CAT_VMAX_THR = 0.06519065318914162
CAT_KCAT_Q   = 0.45

SOD_THR_IC50_UM   = 0.109
SOD_THR_IC50_UGML = 0.7

FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate"
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = [
    "shape", "surface treatment", "dispersion medium", "Substrate"
]


# =========================================================
# 2) 基础函数
# =========================================================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def ensure_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in FEATURE_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    df_raw = ensure_feature_columns(df_raw)
    X = df_raw[FEATURE_COLS].copy()

    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0.0)

    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess_dense():
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop"
    )


def make_label_median_log10(series: pd.Series, mode: str):
    v = pd.to_numeric(series, errors="coerce")
    mask = v.notna() & (v > 0)
    vals = v.loc[mask].astype(float).values
    logv = np.log10(vals)
    thr = float(np.median(logv))

    if mode == "low_good":
        y = (logv <= thr).astype(int)
    else:
        y = (logv >= thr).astype(int)

    return mask, y, thr


def build_labels_log10_single_q(values: pd.Series, task: str, q: float):
    v = pd.to_numeric(values, errors="coerce")
    m = v.notna() & (v > 0)
    vals = np.log10(v.loc[m].astype(float).values)

    if len(vals) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), {
            "scheme": "single_q", "q": q, "thr": np.nan
        }

    if task == "KmLow":
        thr = float(np.quantile(vals, q))
        y = (vals <= thr).astype(int)
    else:
        thr = float(np.quantile(vals, 1 - q))
        y = (vals >= thr).astype(int)

    idx_all = np.where(m.values)[0]
    meta = {"scheme": "single_q", "q": q, "thr": thr}
    return idx_all, y.astype(int), meta


def parse_ic50_cell(x):
    if pd.isna(x):
        return np.nan, np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x), "uM"

    s = str(x).strip().replace("µ", "μ")
    try:
        return float(s), "uM"
    except Exception:
        pass

    m = re.match(r'^\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*(.+?)\s*$', s)
    if not m:
        return np.nan, "unparsed"

    val = float(m.group(1))
    unit = m.group(2).strip().lower().replace(" ", "")
    unit = unit.replace("μ", "u").replace("µ", "u")

    if unit in {"um"}:
        return val, "uM"
    if ("ug" in unit) and ("ml" in unit):
        return val, "ug/mL"

    return np.nan, "unparsed"


def label_sod_ic50(ic50_value, ic50_unit):
    if pd.isna(ic50_value):
        return np.nan
    try:
        v = float(ic50_value)
    except Exception:
        return np.nan
    if v <= 0:
        return np.nan

    if ic50_unit == "uM":
        return int(v <= SOD_THR_IC50_UM)
    if ic50_unit == "ug/mL":
        return int(v <= SOD_THR_IC50_UGML)
    return np.nan


def safe_auc(y_true, p_good):
    try:
        y_true = np.asarray(y_true).astype(int)
        p_good = np.asarray(p_good).astype(float)
        if len(np.unique(y_true)) < 2:
            return None
        return float(roc_auc_score(y_true, p_good))
    except Exception:
        return None


def safe_qcut(s, q=5):
    try:
        return pd.qcut(s, q=q, duplicates="drop")
    except Exception:
        return None


def feature_direction_text(high_mean, low_mean):
    if pd.isna(high_mean) or pd.isna(low_mean):
        return "insufficient variation"
    diff = high_mean - low_mean
    if abs(diff) < 1e-8:
        return "no obvious monotonic trend"
    if diff > 0:
        return "higher values tend to push toward good activity (class 1)"
    return "lower values tend to push toward good activity (class 1)"


# =========================================================
# 3) 为 SHAP 准备数值编码/解码
# =========================================================
def build_shap_numeric_view(X_raw: pd.DataFrame):
    X_num = pd.DataFrame(index=X_raw.index)
    encoders = {}

    for c in NUMERIC_COLS:
        X_num[c] = pd.to_numeric(X_raw[c], errors="coerce").astype(float)

    for c in CATEGORICAL_COLS:
        s = X_raw[c].astype("object")
        s = s.where(~pd.isna(s), "not_reported")
        s = s.astype(str)

        levels = sorted(pd.Series(s).dropna().unique().tolist())
        if "not_reported" not in levels:
            levels = ["not_reported"] + levels

        map_to_code = {lv: i for i, lv in enumerate(levels)}
        code = s.map(map_to_code).fillna(map_to_code["not_reported"]).astype(float)

        X_num[c] = code
        encoders[c] = {
            "levels": levels,
            "map_to_code": map_to_code
        }

    X_num = X_num[FEATURE_COLS].copy()
    return X_num, encoders


def decode_numeric_view_to_raw_df(X_num_like, encoders):
    if isinstance(X_num_like, pd.DataFrame):
        X_num = X_num_like.copy()
    else:
        X_num = pd.DataFrame(X_num_like, columns=FEATURE_COLS)

    X_raw = pd.DataFrame(index=X_num.index)

    for c in NUMERIC_COLS:
        X_raw[c] = pd.to_numeric(X_num[c], errors="coerce")

    for c in CATEGORICAL_COLS:
        levels = encoders[c]["levels"]

        def code_to_level(v):
            if pd.isna(v):
                return "not_reported"
            try:
                iv = int(round(float(v)))
            except Exception:
                return "not_reported"
            if iv < 0 or iv >= len(levels):
                return "not_reported"
            return levels[iv]

        X_raw[c] = X_num[c].apply(code_to_level).astype(object)

    return X_raw[FEATURE_COLS].copy()


# =========================================================
# 4) 六个分类器训练
# =========================================================
def train_cat_km_low(df_cat):
    km = pd.to_numeric(df_cat[COL_KM], errors="coerce")
    m = km.notna() & (km > 0)
    df0 = df_cat.loc[m].copy()
    logkm = np.log10(km.loc[m].astype(float).values)

    keep = (logkm <= CAT_KM_LO) | (logkm >= CAT_KM_HI)
    df_train = df0.iloc[keep].copy()
    y = (logkm[keep] <= CAT_KM_LO).astype(int)

    pipe = Pipeline([
        ("preprocess", make_preprocess_dense()),
        ("model", GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_SEED
        ))
    ])

    X = prepare_features(df_train)
    pipe.fit(X, y)

    info = {
        "task": "CAT_KmLow",
        "train_n": int(len(df_train)),
        "label_rule": "log10(Km)<=lo => 1, log10(Km)>=hi => 0, middle dropped",
        "lo_log10": CAT_KM_LO,
        "hi_log10": CAT_KM_HI,
        "model": "GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)",
        "train_auc_sanity": safe_auc(y, pipe.predict_proba(X)[:, 1]),
    }
    return pipe, X, y, info


def train_cat_vmax_high(df_cat):
    vmax = pd.to_numeric(df_cat[COL_VMAX], errors="coerce")
    mv = vmax.notna() & (vmax > 0)
    df_train = df_cat.loc[mv].copy()
    logv = np.log10(vmax.loc[mv].astype(float).values)
    y = (logv >= CAT_VMAX_THR).astype(int)

    pipe = Pipeline([
        ("preprocess", make_preprocess_dense()),
        ("model", DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_SEED
        ))
    ])

    X = prepare_features(df_train)
    pipe.fit(X, y)

    info = {
        "task": "CAT_VmaxHigh",
        "train_n": int(len(df_train)),
        "label_rule": "log10(Vmax)>=fixed_thr => 1",
        "thr_log10": CAT_VMAX_THR,
        "model": 'DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, class_weight="balanced", random_state=42)',
        "train_auc_sanity": safe_auc(y, pipe.predict_proba(X)[:, 1]),
    }
    return pipe, X, y, info


def train_cat_kcat_high(df_cat):
    idx_keep, y, meta = build_labels_log10_single_q(df_cat[COL_KCAT], task="KcatHigh", q=CAT_KCAT_Q)
    if len(idx_keep) == 0:
        raise RuntimeError("CAT_KcatHigh 无可用样本。")

    df_train = df_cat.iloc[idx_keep].copy()

    pipe = Pipeline([
        ("preprocess", make_preprocess_dense()),
        ("scaler", StandardScaler(with_mean=True)),
        ("model", KNeighborsClassifier(
            n_neighbors=5,
            weights="distance"
        ))
    ])

    X = prepare_features(df_train)
    pipe.fit(X, y)

    info = {
        "task": "CAT_KcatHigh",
        "train_n": int(len(df_train)),
        "label_rule": "single_q(q=0.45) on log10(Kcat), high-good",
        "q": CAT_KCAT_Q,
        "thr_log10": float(meta["thr"]),
        "model": 'KNeighborsClassifier(n_neighbors=5, weights="distance") + StandardScaler',
        "train_auc_sanity": safe_auc(y, pipe.predict_proba(X)[:, 1]),
    }
    return pipe, X, y, info


def train_sod_km_low(df_sod):
    m, y, thr = make_label_median_log10(df_sod[COL_KM], mode="low_good")
    df_train = df_sod.loc[m].copy()

    pipe = Pipeline([
        ("preprocess", make_preprocess_dense()),
        ("scaler", StandardScaler(with_mean=True)),
        ("model", KNeighborsClassifier(
            n_neighbors=5,
            weights="distance"
        ))
    ])

    X = prepare_features(df_train)
    pipe.fit(X, y)

    info = {
        "task": "SOD_KmLow",
        "train_n": int(len(df_train)),
        "label_rule": "median split on log10(Km), low-good",
        "thr_log10": float(thr),
        "model": 'KNeighborsClassifier(n_neighbors=5, weights="distance") + StandardScaler',
        "train_auc_sanity": safe_auc(y, pipe.predict_proba(X)[:, 1]),
    }
    return pipe, X, y, info


def train_sod_vmax_high(df_sod):
    if not HAS_XGBOOST:
        raise RuntimeError("未安装 xgboost，请先执行：pip install xgboost")

    m, y, thr = make_label_median_log10(df_sod[COL_VMAX], mode="high_good")
    df_train = df_sod.loc[m].copy()

    pipe = Pipeline([
        ("preprocess", make_preprocess_dense()),
        ("model", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=0
        ))
    ])

    X = prepare_features(df_train)
    pipe.fit(X, y)

    info = {
        "task": "SOD_VmaxHigh",
        "train_n": int(len(df_train)),
        "label_rule": "median split on log10(Vmax), high-good",
        "thr_log10": float(thr),
        "model": 'XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss", tree_method="hist", random_state=42, n_jobs=-1, verbosity=0)',
        "train_auc_sanity": safe_auc(y, pipe.predict_proba(X)[:, 1]),
    }
    return pipe, X, y, info


def train_sod_ic50_low(df_sod):
    if not HAS_XGBOOST:
        raise RuntimeError("未安装 xgboost，请先执行：pip install xgboost")

    df2 = df_sod.copy()
    parsed = df2[COL_IC50].apply(parse_ic50_cell)
    df2["IC50_value"] = parsed.apply(lambda t: t[0])
    df2["IC50_unit"]  = parsed.apply(lambda t: t[1])
    df2["y_IC50Low"]  = df2.apply(
        lambda r: label_sod_ic50(r["IC50_value"], r["IC50_unit"]), axis=1
    )

    df_train = df2[df2["y_IC50Low"].notna()].copy()
    if df_train.empty:
        raise RuntimeError("SOD_IC50Low 无可用样本。")

    y = df_train["y_IC50Low"].astype(int).values

    pipe = Pipeline([
        ("preprocess", make_preprocess_dense()),
        ("model", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=0
        ))
    ])

    X = prepare_features(df_train)
    pipe.fit(X, y)

    info = {
        "task": "SOD_IC50Low",
        "train_n": int(len(df_train)),
        "label_rule": "IC50<=0.109(uM) or <=0.7(ug/mL) => 1",
        "thr_uM": SOD_THR_IC50_UM,
        "thr_ugmL": SOD_THR_IC50_UGML,
        "model": 'XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss", tree_method="hist", random_state=42, n_jobs=-1, verbosity=0)',
        "train_auc_sanity": safe_auc(y, pipe.predict_proba(X)[:, 1]),
    }
    return pipe, X, y, info


# =========================================================
# 5) SHAP 运行
# =========================================================
def make_predict_fn_from_numeric_view(pipe, encoders):
    def predict_fn(X_num_like):
        X_raw_df = decode_numeric_view_to_raw_df(X_num_like, encoders)
        proba = pipe.predict_proba(X_raw_df)[:, 1]
        return np.asarray(proba, dtype=float).reshape(-1)
    return predict_fn


def save_global_bar(global_imp: pd.DataFrame, out_png: Path, topn=15):
    if global_imp.empty:
        return
    top = global_imp.head(topn).iloc[::-1].copy()

    plt.figure(figsize=(8, max(4, 0.35 * len(top))))
    plt.barh(top["feature"], top["mean_abs_shap"])
    plt.xlabel("Mean |SHAP|")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


def save_top_scatter(X_num: pd.DataFrame, shap_values: np.ndarray, global_imp: pd.DataFrame, out_png: Path, topn=6):
    if global_imp.empty:
        return

    top_feats = global_imp["feature"].head(topn).tolist()
    n = len(top_feats)
    if n == 0:
        return

    fig_h = max(2.2 * n, 4)
    plt.figure(figsize=(8, fig_h))

    for i, feat in enumerate(top_feats, start=1):
        plt.subplot(n, 1, i)
        idx = FEATURE_COLS.index(feat)
        x = pd.to_numeric(X_num[feat], errors="coerce").values
        y = shap_values[:, idx]

        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() > 0:
            jitter = np.random.normal(0, 0.03, size=mask.sum())
            plt.scatter(x[mask], y[mask] + jitter, s=14, alpha=0.7)
        plt.axhline(0, linewidth=0.8)
        plt.xlabel(feat)
        plt.ylabel("SHAP")
        plt.title(feat)

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


def run_one_model_shap(model_name, pipe, X_raw, y, info):
    print(f"[START] {model_name}")

    model_out = OUT_DIR / model_name
    model_out.mkdir(parents=True, exist_ok=True)

    X_raw = X_raw.copy().reset_index(drop=True)
    y = np.asarray(y).astype(int)

    with open(model_out / "model_info.json", "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)

    X_num, encoders = build_shap_numeric_view(X_raw)

    with open(model_out / "categorical_encoders.json", "w", encoding="utf-8") as f:
        json.dump(
            {k: v["levels"] for k, v in encoders.items()},
            f,
            ensure_ascii=False,
            indent=2
        )

    n_bg = min(N_BACKGROUND, len(X_num))
    bg_idx = np.random.choice(len(X_num), size=n_bg, replace=False)
    X_bg = X_num.iloc[bg_idx].copy()

    predict_fn = make_predict_fn_from_numeric_view(pipe, encoders)

    explainer = shap.Explainer(
        predict_fn,
        X_bg,
        algorithm="permutation"
    )

    max_evals = max(2 * X_num.shape[1] + 1, MAX_EVALS_FACTOR * X_num.shape[1] + 1)
    shap_exp = explainer(X_num, max_evals=max_evals)

    shap_values = np.asarray(shap_exp.values, dtype=float)
    if shap_values.ndim == 3:
        shap_values = np.squeeze(shap_values)
    if shap_values.ndim != 2:
        raise RuntimeError(f"{model_name}: Unexpected SHAP shape: {shap_values.shape}")

    pred = predict_fn(X_num)

    # ---------------- sample-level ----------------
    sample_out = pd.DataFrame(index=np.arange(len(X_raw)))
    sample_out["model_name"] = model_name
    sample_out["y_true"] = y
    sample_out["p_good"] = pred

    for c in FEATURE_COLS:
        sample_out[c] = X_raw[c].values

    for i, c in enumerate(FEATURE_COLS):
        sample_out[f"SHAP__{c}"] = shap_values[:, i]

    sample_out.to_csv(model_out / "sample_level_shap.csv", index=False, encoding="utf-8-sig")

    # ---------------- global importance ----------------
    global_imp = pd.DataFrame({
        "model_name": model_name,
        "feature": FEATURE_COLS,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0),
        "mean_shap": shap_values.mean(axis=0),
        "shap_std": shap_values.std(axis=0),
        "rank_within_model": pd.Series(np.abs(shap_values).mean(axis=0)).rank(ascending=False, method="dense").astype(int)
    }).sort_values("mean_abs_shap", ascending=False)

    global_imp.to_csv(model_out / "global_importance.csv", index=False, encoding="utf-8-sig")

    # ---------------- numeric feature summary ----------------
    numeric_rows = []
    numeric_bin_rows = []

    for c in NUMERIC_COLS:
        s = pd.to_numeric(X_raw[c], errors="coerce")
        sv = shap_values[:, FEATURE_COLS.index(c)]

        ok = s.notna()
        if ok.sum() < 5:
            continue

        s_ok = s[ok]
        sv_ok = sv[ok.values]

        q25 = s_ok.quantile(0.25)
        q75 = s_ok.quantile(0.75)

        low_mask = s_ok <= q25
        high_mask = s_ok >= q75

        low_mean_shap = float(np.mean(sv_ok[low_mask])) if low_mask.sum() > 0 else np.nan
        high_mean_shap = float(np.mean(sv_ok[high_mask])) if high_mask.sum() > 0 else np.nan

        if s_ok.std() > 1e-12 and np.std(sv_ok) > 1e-12:
            corr_val_shap = float(np.corrcoef(s_ok.values, sv_ok)[0, 1])
        else:
            corr_val_shap = np.nan

        numeric_rows.append({
            "model_name": model_name,
            "feature": c,
            "n": int(ok.sum()),
            "mean_abs_shap": float(np.mean(np.abs(sv_ok))),
            "mean_shap": float(np.mean(sv_ok)),
            "value_min": float(np.min(s_ok)),
            "value_q25": float(q25),
            "value_median": float(np.median(s_ok)),
            "value_q75": float(q75),
            "value_max": float(np.max(s_ok)),
            "corr_value_shap": corr_val_shap,
            "mean_shap_low25": low_mean_shap,
            "mean_shap_high25": high_mean_shap,
            "direction_hint": feature_direction_text(high_mean_shap, low_mean_shap),
        })

        bins = safe_qcut(s_ok, q=N_BINS_NUMERIC)
        if bins is not None:
            tmp = pd.DataFrame({
                "feature_value": s_ok.values,
                "shap_value": sv_ok,
                "bin": bins.astype(str).values
            })
            for bin_name, sub in tmp.groupby("bin", dropna=False):
                numeric_bin_rows.append({
                    "model_name": model_name,
                    "feature": c,
                    "bin": str(bin_name),
                    "n": int(len(sub)),
                    "feature_min": float(sub["feature_value"].min()),
                    "feature_max": float(sub["feature_value"].max()),
                    "feature_mean": float(sub["feature_value"].mean()),
                    "shap_mean": float(sub["shap_value"].mean()),
                    "shap_median": float(sub["shap_value"].median()),
                    "shap_mean_abs": float(np.abs(sub["shap_value"]).mean()),
                })

    numeric_summary = pd.DataFrame(numeric_rows).sort_values(["model_name", "mean_abs_shap"], ascending=[True, False])
    numeric_summary.to_csv(model_out / "numeric_feature_summary.csv", index=False, encoding="utf-8-sig")

    numeric_bins = pd.DataFrame(numeric_bin_rows)
    if not numeric_bins.empty:
        numeric_bins = numeric_bins.sort_values(["model_name", "feature", "feature_mean"], ascending=[True, True, True])
    numeric_bins.to_csv(model_out / "numeric_feature_bins.csv", index=False, encoding="utf-8-sig")

    # ---------------- categorical feature summary ----------------
    cat_rows = []
    for c in CATEGORICAL_COLS:
        s = X_raw[c].astype("object")
        s = s.where(~pd.isna(s), "not_reported")
        s = s.astype(str)
        sv = shap_values[:, FEATURE_COLS.index(c)]

        tmp = pd.DataFrame({
            "level": s.values,
            "shap_value": sv
        })

        for level, sub in tmp.groupby("level", dropna=False):
            cat_rows.append({
                "model_name": model_name,
                "feature": c,
                "level": str(level),
                "n": int(len(sub)),
                "mean_shap": float(sub["shap_value"].mean()),
                "median_shap": float(sub["shap_value"].median()),
                "mean_abs_shap": float(np.abs(sub["shap_value"]).mean()),
            })

    cat_summary = pd.DataFrame(cat_rows)
    if not cat_summary.empty:
        cat_summary = cat_summary.sort_values(["model_name", "feature", "mean_abs_shap"], ascending=[True, True, False])
    cat_summary.to_csv(model_out / "categorical_feature_levels.csv", index=False, encoding="utf-8-sig")

    # ---------------- plots ----------------
    save_global_bar(global_imp, model_out / "global_importance_bar.png", topn=TOPN_PLOT)
    save_top_scatter(X_num, shap_values, global_imp, model_out / "top_features_scatter.png", topn=min(6, TOPN_PLOT))

    print(f"[OK] {model_name} done.")

    return {
        "model_info": pd.DataFrame([info]),
        "global_importance": global_imp,
        "numeric_feature_summary": numeric_summary,
        "numeric_feature_bins": numeric_bins,
        "categorical_feature_levels": cat_summary
    }


# =========================================================
# 6) 汇总导出
# =========================================================
def export_combined_tables(combined_results: list):
    model_info_all = []
    global_imp_all = []
    numeric_summary_all = []
    numeric_bins_all = []
    categorical_all = []

    for res in combined_results:
        model_info_all.append(res["model_info"])
        global_imp_all.append(res["global_importance"])
        numeric_summary_all.append(res["numeric_feature_summary"])
        numeric_bins_all.append(res["numeric_feature_bins"])
        categorical_all.append(res["categorical_feature_levels"])

    df_model_info = pd.concat(model_info_all, ignore_index=True) if model_info_all else pd.DataFrame()
    df_global = pd.concat(global_imp_all, ignore_index=True) if global_imp_all else pd.DataFrame()
    df_numeric_summary = pd.concat(numeric_summary_all, ignore_index=True) if numeric_summary_all else pd.DataFrame()
    df_numeric_bins = pd.concat(numeric_bins_all, ignore_index=True) if numeric_bins_all else pd.DataFrame()
    df_categorical = pd.concat(categorical_all, ignore_index=True) if categorical_all else pd.DataFrame()

    if not df_global.empty:
        df_global = df_global.sort_values(["model_name", "mean_abs_shap"], ascending=[True, False])

    if not df_numeric_summary.empty:
        df_numeric_summary = df_numeric_summary.sort_values(["model_name", "mean_abs_shap"], ascending=[True, False])

    if not df_numeric_bins.empty:
        df_numeric_bins = df_numeric_bins.sort_values(["model_name", "feature", "feature_mean"], ascending=[True, True, True])

    if not df_categorical.empty:
        df_categorical = df_categorical.sort_values(["model_name", "feature", "mean_abs_shap"], ascending=[True, True, False])

    # CSV
    df_model_info.to_csv(OUT_DIR / "combined_model_info.csv", index=False, encoding="utf-8-sig")
    df_global.to_csv(OUT_DIR / "combined_global_importance.csv", index=False, encoding="utf-8-sig")
    df_numeric_summary.to_csv(OUT_DIR / "combined_numeric_feature_summary.csv", index=False, encoding="utf-8-sig")
    df_numeric_bins.to_csv(OUT_DIR / "combined_numeric_feature_bins.csv", index=False, encoding="utf-8-sig")
    df_categorical.to_csv(OUT_DIR / "combined_categorical_feature_levels.csv", index=False, encoding="utf-8-sig")

    # Excel 总汇总
    xlsx_path = OUT_DIR / "all_models_summary.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df_model_info.to_excel(writer, sheet_name="model_info", index=False)
        df_global.to_excel(writer, sheet_name="global_importance", index=False)
        df_numeric_summary.to_excel(writer, sheet_name="numeric_summary", index=False)
        df_numeric_bins.to_excel(writer, sheet_name="numeric_bins", index=False)
        df_categorical.to_excel(writer, sheet_name="categorical_levels", index=False)

    print("=" * 80)
    print("Combined tables saved:")
    print(OUT_DIR / "combined_model_info.csv")
    print(OUT_DIR / "combined_global_importance.csv")
    print(OUT_DIR / "combined_numeric_feature_summary.csv")
    print(OUT_DIR / "combined_numeric_feature_bins.csv")
    print(OUT_DIR / "combined_categorical_feature_levels.csv")
    print(OUT_DIR / "all_models_summary.xlsx")
    print("=" * 80)


# =========================================================
# 7) 主程序
# =========================================================
def main():
    np.random.seed(RANDOM_SEED)

    print("=" * 80)
    print("Loading data...")
    print("CAT_PATH:", CAT_PATH)
    print("SOD_PATH:", SOD_PATH)
    print("OUT_DIR :", OUT_DIR)
    print("=" * 80)

    if not CAT_PATH.exists():
        raise FileNotFoundError(f"CAT file not found: {CAT_PATH}")
    if not SOD_PATH.exists():
        raise FileNotFoundError(f"SOD file not found: {SOD_PATH}")

    df_cat = normalize_columns(pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME))
    df_sod = normalize_columns(pd.read_excel(SOD_PATH, sheet_name=SHEET_NAME))

    jobs = []

    pipe, X, y, info = train_cat_km_low(df_cat)
    jobs.append(("CAT_KmLow", pipe, X, y, info))

    pipe, X, y, info = train_cat_vmax_high(df_cat)
    jobs.append(("CAT_VmaxHigh", pipe, X, y, info))

    pipe, X, y, info = train_cat_kcat_high(df_cat)
    jobs.append(("CAT_KcatHigh", pipe, X, y, info))

    pipe, X, y, info = train_sod_km_low(df_sod)
    jobs.append(("SOD_KmLow", pipe, X, y, info))

    pipe, X, y, info = train_sod_vmax_high(df_sod)
    jobs.append(("SOD_VmaxHigh", pipe, X, y, info))

    pipe, X, y, info = train_sod_ic50_low(df_sod)
    jobs.append(("SOD_IC50Low", pipe, X, y, info))

    combined_results = []

    for model_name, pipe, X, y, info in jobs:
        res = run_one_model_shap(model_name, pipe, X, y, info)
        combined_results.append(res)

    export_combined_tables(combined_results)

    with open(OUT_DIR / "README_interpretation.txt", "w", encoding="utf-8") as f:
        f.write(
            "解释规则：\n"
            "1) 所有任务都按 class 1 = 好活性。\n"
            "2) SHAP > 0 表示该特征把样本推向好活性。\n"
            "3) 最建议直接上传 all_models_summary.xlsx。\n"
            "4) 如果传 CSV，优先传：combined_global_importance.csv、combined_numeric_feature_summary.csv、combined_numeric_feature_bins.csv、combined_categorical_feature_levels.csv。\n"
            "5) main metal number 这类变量，最好结合 numeric_bins 说具体原子序数区间，而不是只说较轻/较重。\n"
        )

    print("=" * 80)
    print("All done.")
    print("Results saved to:", OUT_DIR)
    print("=" * 80)


if __name__ == "__main__":
    main()

Loading data...
CAT_PATH: C:\Users\86158\Desktop\dataiscat.xlsx
SOD_PATH: C:\Users\86158\Desktop\dataissod.xlsx
OUT_DIR : C:\Users\86158\Desktop\all6_classifier_raw_shap_summary
[START] CAT_KmLow
[OK] CAT_KmLow done.
[START] CAT_VmaxHigh
[OK] CAT_VmaxHigh done.
[START] CAT_KcatHigh
[OK] CAT_KcatHigh done.
[START] SOD_KmLow


PermutationExplainer explainer: 20it [00:11,  3.89s/it]                                                                


[OK] SOD_KmLow done.
[START] SOD_VmaxHigh
[OK] SOD_VmaxHigh done.
[START] SOD_IC50Low
[OK] SOD_IC50Low done.
Combined tables saved:
C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\combined_model_info.csv
C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\combined_global_importance.csv
C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\combined_numeric_feature_summary.csv
C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\combined_numeric_feature_bins.csv
C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\combined_categorical_feature_levels.csv
C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\all_models_summary.xlsx
All done.
Results saved to: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary


In [10]:
# -*- coding: utf-8 -*-
"""
把 main/minor metal number 翻译成具体元素，并做元素级汇总（修复完整版）
====================================================================
输入：
- 你之前 SHAP 稳定汇总脚本生成的模型子目录
  例如：
  C:\\Users\\86158\\Desktop\\all6_classifier_raw_shap_summary\\CAT_KmLow\\sample_level_shap.csv
  C:\\Users\\86158\\Desktop\\all6_classifier_raw_shap_summary\\SOD_IC50Low\\sample_level_shap.csv
  ...

输出：
- element_level_summary/
    - main_metal_element_summary_by_model.csv
    - minor_metal_element_summary_by_model.csv
    - metal_pair_summary_by_model.csv
    - all_element_level_summary.xlsx
    - README_element_summary.json

说明：
- 所有任务都按 class 1 = 好活性
- 因此 mean_shap > 0 表示该元素/组合整体更推向好活性
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd


# =========================================================
# 0) 路径
# =========================================================
BASE_DIR = Path(r"C:\Users\86158\Desktop\all6_classifier_raw_shap_summary")
OUT_DIR = BASE_DIR / "element_level_summary"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# 1) 周期表信息
# =========================================================
PERIODIC_TABLE = {
     1:  {"symbol":"H",  "name":"Hydrogen",      "period":1, "group":1,   "block":"s", "family":"nonmetal"},
     2:  {"symbol":"He", "name":"Helium",        "period":1, "group":18,  "block":"s", "family":"noble gas"},
     3:  {"symbol":"Li", "name":"Lithium",       "period":2, "group":1,   "block":"s", "family":"alkali metal"},
     4:  {"symbol":"Be", "name":"Beryllium",     "period":2, "group":2,   "block":"s", "family":"alkaline earth metal"},
     5:  {"symbol":"B",  "name":"Boron",         "period":2, "group":13,  "block":"p", "family":"metalloid"},
     6:  {"symbol":"C",  "name":"Carbon",        "period":2, "group":14,  "block":"p", "family":"nonmetal"},
     7:  {"symbol":"N",  "name":"Nitrogen",      "period":2, "group":15,  "block":"p", "family":"nonmetal"},
     8:  {"symbol":"O",  "name":"Oxygen",        "period":2, "group":16,  "block":"p", "family":"nonmetal"},
     9:  {"symbol":"F",  "name":"Fluorine",      "period":2, "group":17,  "block":"p", "family":"halogen"},
    10:  {"symbol":"Ne", "name":"Neon",          "period":2, "group":18,  "block":"p", "family":"noble gas"},

    11:  {"symbol":"Na", "name":"Sodium",        "period":3, "group":1,   "block":"s", "family":"alkali metal"},
    12:  {"symbol":"Mg", "name":"Magnesium",     "period":3, "group":2,   "block":"s", "family":"alkaline earth metal"},
    13:  {"symbol":"Al", "name":"Aluminium",     "period":3, "group":13,  "block":"p", "family":"post-transition metal"},
    14:  {"symbol":"Si", "name":"Silicon",       "period":3, "group":14,  "block":"p", "family":"metalloid"},
    15:  {"symbol":"P",  "name":"Phosphorus",    "period":3, "group":15,  "block":"p", "family":"nonmetal"},
    16:  {"symbol":"S",  "name":"Sulfur",        "period":3, "group":16,  "block":"p", "family":"nonmetal"},
    17:  {"symbol":"Cl", "name":"Chlorine",      "period":3, "group":17,  "block":"p", "family":"halogen"},
    18:  {"symbol":"Ar", "name":"Argon",         "period":3, "group":18,  "block":"p", "family":"noble gas"},

    19:  {"symbol":"K",  "name":"Potassium",     "period":4, "group":1,   "block":"s", "family":"alkali metal"},
    20:  {"symbol":"Ca", "name":"Calcium",       "period":4, "group":2,   "block":"s", "family":"alkaline earth metal"},
    21:  {"symbol":"Sc", "name":"Scandium",      "period":4, "group":3,   "block":"d", "family":"transition metal"},
    22:  {"symbol":"Ti", "name":"Titanium",      "period":4, "group":4,   "block":"d", "family":"transition metal"},
    23:  {"symbol":"V",  "name":"Vanadium",      "period":4, "group":5,   "block":"d", "family":"transition metal"},
    24:  {"symbol":"Cr", "name":"Chromium",      "period":4, "group":6,   "block":"d", "family":"transition metal"},
    25:  {"symbol":"Mn", "name":"Manganese",     "period":4, "group":7,   "block":"d", "family":"transition metal"},
    26:  {"symbol":"Fe", "name":"Iron",          "period":4, "group":8,   "block":"d", "family":"transition metal"},
    27:  {"symbol":"Co", "name":"Cobalt",        "period":4, "group":9,   "block":"d", "family":"transition metal"},
    28:  {"symbol":"Ni", "name":"Nickel",        "period":4, "group":10,  "block":"d", "family":"transition metal"},
    29:  {"symbol":"Cu", "name":"Copper",        "period":4, "group":11,  "block":"d", "family":"transition metal"},
    30:  {"symbol":"Zn", "name":"Zinc",          "period":4, "group":12,  "block":"d", "family":"transition metal"},
    31:  {"symbol":"Ga", "name":"Gallium",       "period":4, "group":13,  "block":"p", "family":"post-transition metal"},
    32:  {"symbol":"Ge", "name":"Germanium",     "period":4, "group":14,  "block":"p", "family":"metalloid"},
    33:  {"symbol":"As", "name":"Arsenic",       "period":4, "group":15,  "block":"p", "family":"metalloid"},
    34:  {"symbol":"Se", "name":"Selenium",      "period":4, "group":16,  "block":"p", "family":"nonmetal"},
    35:  {"symbol":"Br", "name":"Bromine",       "period":4, "group":17,  "block":"p", "family":"halogen"},
    36:  {"symbol":"Kr", "name":"Krypton",       "period":4, "group":18,  "block":"p", "family":"noble gas"},

    37:  {"symbol":"Rb", "name":"Rubidium",      "period":5, "group":1,   "block":"s", "family":"alkali metal"},
    38:  {"symbol":"Sr", "name":"Strontium",     "period":5, "group":2,   "block":"s", "family":"alkaline earth metal"},
    39:  {"symbol":"Y",  "name":"Yttrium",       "period":5, "group":3,   "block":"d", "family":"transition metal"},
    40:  {"symbol":"Zr", "name":"Zirconium",     "period":5, "group":4,   "block":"d", "family":"transition metal"},
    41:  {"symbol":"Nb", "name":"Niobium",       "period":5, "group":5,   "block":"d", "family":"transition metal"},
    42:  {"symbol":"Mo", "name":"Molybdenum",    "period":5, "group":6,   "block":"d", "family":"transition metal"},
    43:  {"symbol":"Tc", "name":"Technetium",    "period":5, "group":7,   "block":"d", "family":"transition metal"},
    44:  {"symbol":"Ru", "name":"Ruthenium",     "period":5, "group":8,   "block":"d", "family":"transition metal"},
    45:  {"symbol":"Rh", "name":"Rhodium",       "period":5, "group":9,   "block":"d", "family":"transition metal"},
    46:  {"symbol":"Pd", "name":"Palladium",     "period":5, "group":10,  "block":"d", "family":"transition metal"},
    47:  {"symbol":"Ag", "name":"Silver",        "period":5, "group":11,  "block":"d", "family":"transition metal"},
    48:  {"symbol":"Cd", "name":"Cadmium",       "period":5, "group":12,  "block":"d", "family":"transition metal"},
    49:  {"symbol":"In", "name":"Indium",        "period":5, "group":13,  "block":"p", "family":"post-transition metal"},
    50:  {"symbol":"Sn", "name":"Tin",           "period":5, "group":14,  "block":"p", "family":"post-transition metal"},
    51:  {"symbol":"Sb", "name":"Antimony",      "period":5, "group":15,  "block":"p", "family":"metalloid"},
    52:  {"symbol":"Te", "name":"Tellurium",     "period":5, "group":16,  "block":"p", "family":"metalloid"},
    53:  {"symbol":"I",  "name":"Iodine",        "period":5, "group":17,  "block":"p", "family":"halogen"},
    54:  {"symbol":"Xe", "name":"Xenon",         "period":5, "group":18,  "block":"p", "family":"noble gas"},

    55:  {"symbol":"Cs", "name":"Caesium",       "period":6, "group":1,   "block":"s", "family":"alkali metal"},
    56:  {"symbol":"Ba", "name":"Barium",        "period":6, "group":2,   "block":"s", "family":"alkaline earth metal"},
    57:  {"symbol":"La", "name":"Lanthanum",     "period":6, "group":3,   "block":"d", "family":"lanthanide-related"},
    58:  {"symbol":"Ce", "name":"Cerium",        "period":6, "group":None,"block":"f", "family":"lanthanide"},
    59:  {"symbol":"Pr", "name":"Praseodymium",  "period":6, "group":None,"block":"f", "family":"lanthanide"},
    60:  {"symbol":"Nd", "name":"Neodymium",     "period":6, "group":None,"block":"f", "family":"lanthanide"},
    61:  {"symbol":"Pm", "name":"Promethium",    "period":6, "group":None,"block":"f", "family":"lanthanide"},
    62:  {"symbol":"Sm", "name":"Samarium",      "period":6, "group":None,"block":"f", "family":"lanthanide"},
    63:  {"symbol":"Eu", "name":"Europium",      "period":6, "group":None,"block":"f", "family":"lanthanide"},
    64:  {"symbol":"Gd", "name":"Gadolinium",    "period":6, "group":None,"block":"f", "family":"lanthanide"},
    65:  {"symbol":"Tb", "name":"Terbium",       "period":6, "group":None,"block":"f", "family":"lanthanide"},
    66:  {"symbol":"Dy", "name":"Dysprosium",    "period":6, "group":None,"block":"f", "family":"lanthanide"},
    67:  {"symbol":"Ho", "name":"Holmium",       "period":6, "group":None,"block":"f", "family":"lanthanide"},
    68:  {"symbol":"Er", "name":"Erbium",        "period":6, "group":None,"block":"f", "family":"lanthanide"},
    69:  {"symbol":"Tm", "name":"Thulium",       "period":6, "group":None,"block":"f", "family":"lanthanide"},
    70:  {"symbol":"Yb", "name":"Ytterbium",     "period":6, "group":None,"block":"f", "family":"lanthanide"},
    71:  {"symbol":"Lu", "name":"Lutetium",      "period":6, "group":3,   "block":"d", "family":"lanthanide-related"},
    72:  {"symbol":"Hf", "name":"Hafnium",       "period":6, "group":4,   "block":"d", "family":"transition metal"},
    73:  {"symbol":"Ta", "name":"Tantalum",      "period":6, "group":5,   "block":"d", "family":"transition metal"},
    74:  {"symbol":"W",  "name":"Tungsten",      "period":6, "group":6,   "block":"d", "family":"transition metal"},
    75:  {"symbol":"Re", "name":"Rhenium",       "period":6, "group":7,   "block":"d", "family":"transition metal"},
    76:  {"symbol":"Os", "name":"Osmium",        "period":6, "group":8,   "block":"d", "family":"transition metal"},
    77:  {"symbol":"Ir", "name":"Iridium",       "period":6, "group":9,   "block":"d", "family":"transition metal"},
    78:  {"symbol":"Pt", "name":"Platinum",      "period":6, "group":10,  "block":"d", "family":"transition metal"},
    79:  {"symbol":"Au", "name":"Gold",          "period":6, "group":11,  "block":"d", "family":"transition metal"},
    80:  {"symbol":"Hg", "name":"Mercury",       "period":6, "group":12,  "block":"d", "family":"transition metal"},
    81:  {"symbol":"Tl", "name":"Thallium",      "period":6, "group":13,  "block":"p", "family":"post-transition metal"},
    82:  {"symbol":"Pb", "name":"Lead",          "period":6, "group":14,  "block":"p", "family":"post-transition metal"},
    83:  {"symbol":"Bi", "name":"Bismuth",       "period":6, "group":15,  "block":"p", "family":"post-transition metal"},
    84:  {"symbol":"Po", "name":"Polonium",      "period":6, "group":16,  "block":"p", "family":"post-transition metal"},
    85:  {"symbol":"At", "name":"Astatine",      "period":6, "group":17,  "block":"p", "family":"halogen"},
    86:  {"symbol":"Rn", "name":"Radon",         "period":6, "group":18,  "block":"p", "family":"noble gas"},

    87:  {"symbol":"Fr", "name":"Francium",      "period":7, "group":1,   "block":"s", "family":"alkali metal"},
    88:  {"symbol":"Ra", "name":"Radium",        "period":7, "group":2,   "block":"s", "family":"alkaline earth metal"},
    89:  {"symbol":"Ac", "name":"Actinium",      "period":7, "group":3,   "block":"d", "family":"actinide-related"},
    90:  {"symbol":"Th", "name":"Thorium",       "period":7, "group":None,"block":"f", "family":"actinide"},
    91:  {"symbol":"Pa", "name":"Protactinium",  "period":7, "group":None,"block":"f", "family":"actinide"},
    92:  {"symbol":"U",  "name":"Uranium",       "period":7, "group":None,"block":"f", "family":"actinide"},
    93:  {"symbol":"Np", "name":"Neptunium",     "period":7, "group":None,"block":"f", "family":"actinide"},
    94:  {"symbol":"Pu", "name":"Plutonium",     "period":7, "group":None,"block":"f", "family":"actinide"},
    95:  {"symbol":"Am", "name":"Americium",     "period":7, "group":None,"block":"f", "family":"actinide"},
    96:  {"symbol":"Cm", "name":"Curium",        "period":7, "group":None,"block":"f", "family":"actinide"},
    97:  {"symbol":"Bk", "name":"Berkelium",     "period":7, "group":None,"block":"f", "family":"actinide"},
    98:  {"symbol":"Cf", "name":"Californium",   "period":7, "group":None,"block":"f", "family":"actinide"},
    99:  {"symbol":"Es", "name":"Einsteinium",   "period":7, "group":None,"block":"f", "family":"actinide"},
    100: {"symbol":"Fm", "name":"Fermium",       "period":7, "group":None,"block":"f", "family":"actinide"},
    101: {"symbol":"Md", "name":"Mendelevium",   "period":7, "group":None,"block":"f", "family":"actinide"},
    102: {"symbol":"No", "name":"Nobelium",      "period":7, "group":None,"block":"f", "family":"actinide"},
    103: {"symbol":"Lr", "name":"Lawrencium",    "period":7, "group":3,   "block":"d", "family":"actinide-related"},
    104: {"symbol":"Rf", "name":"Rutherfordium", "period":7, "group":4,   "block":"d", "family":"transition metal"},
    105: {"symbol":"Db", "name":"Dubnium",       "period":7, "group":5,   "block":"d", "family":"transition metal"},
    106: {"symbol":"Sg", "name":"Seaborgium",    "period":7, "group":6,   "block":"d", "family":"transition metal"},
    107: {"symbol":"Bh", "name":"Bohrium",       "period":7, "group":7,   "block":"d", "family":"transition metal"},
    108: {"symbol":"Hs", "name":"Hassium",       "period":7, "group":8,   "block":"d", "family":"transition metal"},
    109: {"symbol":"Mt", "name":"Meitnerium",    "period":7, "group":9,   "block":"d", "family":"transition metal"},
    110: {"symbol":"Ds", "name":"Darmstadtium",  "period":7, "group":10,  "block":"d", "family":"transition metal"},
    111: {"symbol":"Rg", "name":"Roentgenium",   "period":7, "group":11,  "block":"d", "family":"transition metal"},
    112: {"symbol":"Cn", "name":"Copernicium",   "period":7, "group":12,  "block":"d", "family":"transition metal"},
    113: {"symbol":"Nh", "name":"Nihonium",      "period":7, "group":13,  "block":"p", "family":"post-transition metal"},
    114: {"symbol":"Fl", "name":"Flerovium",     "period":7, "group":14,  "block":"p", "family":"post-transition metal"},
    115: {"symbol":"Mc", "name":"Moscovium",     "period":7, "group":15,  "block":"p", "family":"post-transition metal"},
    116: {"symbol":"Lv", "name":"Livermorium",   "period":7, "group":16,  "block":"p", "family":"post-transition metal"},
    117: {"symbol":"Ts", "name":"Tennessine",    "period":7, "group":17,  "block":"p", "family":"halogen"},
    118: {"symbol":"Og", "name":"Oganesson",     "period":7, "group":18,  "block":"p", "family":"noble gas"},
}


# =========================================================
# 2) 工具函数
# =========================================================
def drop_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """删除重复列名，只保留第一次出现的列"""
    return df.loc[:, ~df.columns.duplicated()].copy()


def safe_round_atomic_number(x):
    if pd.isna(x):
        return np.nan
    try:
        v = float(x)
    except Exception:
        return np.nan
    if not np.isfinite(v):
        return np.nan
    if abs(v) < 1e-12:
        return 0
    return int(round(v))


def atomic_to_info(z):
    """
    0 -> no metal / not reported
    1~118 -> 周期表信息
    other -> unknown
    """
    if pd.isna(z):
        return {
            "atomic_number": np.nan,
            "element_symbol": np.nan,
            "element_name": np.nan,
            "period": np.nan,
            "group": np.nan,
            "block": np.nan,
            "family": np.nan,
        }

    z = safe_round_atomic_number(z)

    if z == 0:
        return {
            "atomic_number": 0,
            "element_symbol": "None",
            "element_name": "No secondary metal / not reported",
            "period": np.nan,
            "group": np.nan,
            "block": np.nan,
            "family": "none",
        }

    if z in PERIODIC_TABLE:
        d = PERIODIC_TABLE[z]
        return {
            "atomic_number": z,
            "element_symbol": d["symbol"],
            "element_name": d["name"],
            "period": d["period"],
            "group": d["group"],
            "block": d["block"],
            "family": d["family"],
        }

    return {
        "atomic_number": z,
        "element_symbol": "Unknown",
        "element_name": "Unknown",
        "period": np.nan,
        "group": np.nan,
        "block": np.nan,
        "family": "unknown",
    }


def summarize_one_role(df, model_name, metal_col, shap_col, role_name):
    """
    对主金属或辅金属单独做元素级汇总
    """
    need_cols = [metal_col, shap_col, "p_good", "y_true"]
    for c in need_cols:
        if c not in df.columns:
            return pd.DataFrame()

    tmp = df[need_cols].copy()
    tmp["atomic_number_raw"] = tmp[metal_col].apply(safe_round_atomic_number)

    # 翻译元素信息
    info_df = tmp["atomic_number_raw"].apply(atomic_to_info).apply(pd.Series)

    # 合并后去重保险
    tmp = pd.concat([tmp, info_df], axis=1)
    tmp = drop_duplicate_columns(tmp)

    # 用翻译后的 atomic_number 作为正式列
    tmp = tmp[tmp["atomic_number"].notna()].copy()
    if tmp.empty:
        return pd.DataFrame()

    g = tmp.groupby(
        ["atomic_number", "element_symbol", "element_name", "period", "group", "block", "family"],
        dropna=False
    )

    out = g.agg(
        n=("atomic_number", "size"),
        mean_shap=(shap_col, "mean"),
        median_shap=(shap_col, "median"),
        mean_abs_shap=(shap_col, lambda s: np.mean(np.abs(s))),
        pos_shap_rate=(shap_col, lambda s: np.mean(np.asarray(s) > 0)),
        mean_p_good=("p_good", "mean"),
        pos_class_rate=("y_true", "mean"),
    ).reset_index()

    out.insert(0, "model_name", model_name)
    out.insert(1, "metal_role", role_name)
    out = out.sort_values(["mean_shap", "mean_abs_shap", "n"], ascending=[False, False, False])

    return out


def summarize_pairs(df, model_name):
    """
    汇总 主金属-辅金属 组合
    """
    need_cols = [
        "main metal number", "minor metal number",
        "SHAP__main metal number", "SHAP__minor metal number",
        "p_good", "y_true"
    ]
    for c in need_cols:
        if c not in df.columns:
            return pd.DataFrame()

    tmp = df[need_cols].copy()
    tmp["main_atomic_number_raw"] = tmp["main metal number"].apply(safe_round_atomic_number)
    tmp["minor_atomic_number_raw"] = tmp["minor metal number"].apply(safe_round_atomic_number)

    main_info = tmp["main_atomic_number_raw"].apply(atomic_to_info).apply(pd.Series)
    minor_info = tmp["minor_atomic_number_raw"].apply(atomic_to_info).apply(pd.Series)

    # 加前缀，避免和 raw 列重名
    main_info = main_info.add_prefix("main_")
    minor_info = minor_info.add_prefix("minor_")

    tmp = pd.concat([tmp, main_info, minor_info], axis=1)
    tmp = drop_duplicate_columns(tmp)

    tmp["pair_shap_sum"] = tmp["SHAP__main metal number"] + tmp["SHAP__minor metal number"]
    tmp["pair_shap_abs_sum"] = np.abs(tmp["SHAP__main metal number"]) + np.abs(tmp["SHAP__minor metal number"])

    tmp = tmp[tmp["main_atomic_number"].notna()].copy()
    if tmp.empty:
        return pd.DataFrame()

    g = tmp.groupby(
        [
            "main_atomic_number", "main_element_symbol", "main_element_name",
            "minor_atomic_number", "minor_element_symbol", "minor_element_name"
        ],
        dropna=False
    )

    out = g.agg(
        n=("main_atomic_number", "size"),
        mean_pair_shap=("pair_shap_sum", "mean"),
        median_pair_shap=("pair_shap_sum", "median"),
        mean_pair_abs_shap=("pair_shap_abs_sum", "mean"),
        mean_main_shap=("SHAP__main metal number", "mean"),
        mean_minor_shap=("SHAP__minor metal number", "mean"),
        mean_p_good=("p_good", "mean"),
        pos_class_rate=("y_true", "mean"),
    ).reset_index()

    out.insert(0, "model_name", model_name)
    out = out.sort_values(["mean_pair_shap", "mean_pair_abs_shap", "n"], ascending=[False, False, False])

    return out


# =========================================================
# 3) 主程序
# =========================================================
def main():
    model_dirs = [p for p in BASE_DIR.iterdir() if p.is_dir()]
    model_dirs = [p for p in model_dirs if (p / "sample_level_shap.csv").exists()]

    if not model_dirs:
        raise FileNotFoundError(
            f"在 {BASE_DIR} 下没有找到任何包含 sample_level_shap.csv 的模型子目录。"
        )

    main_tables = []
    minor_tables = []
    pair_tables = []

    for mdir in sorted(model_dirs):
        model_name = mdir.name
        csv_path = mdir / "sample_level_shap.csv"
        print(f"[READ] {model_name}: {csv_path}")

        df = pd.read_csv(csv_path, encoding="utf-8-sig")
        df = drop_duplicate_columns(df)

        # 主金属
        if "main metal number" in df.columns and "SHAP__main metal number" in df.columns:
            t_main = summarize_one_role(
                df=df,
                model_name=model_name,
                metal_col="main metal number",
                shap_col="SHAP__main metal number",
                role_name="main"
            )
            if not t_main.empty:
                main_tables.append(t_main)

        # 辅金属
        if "minor metal number" in df.columns and "SHAP__minor metal number" in df.columns:
            t_minor = summarize_one_role(
                df=df,
                model_name=model_name,
                metal_col="minor metal number",
                shap_col="SHAP__minor metal number",
                role_name="minor"
            )
            if not t_minor.empty:
                minor_tables.append(t_minor)

        # 主-辅组合
        t_pair = summarize_pairs(df, model_name)
        if not t_pair.empty:
            pair_tables.append(t_pair)

    df_main = pd.concat(main_tables, ignore_index=True) if main_tables else pd.DataFrame()
    df_minor = pd.concat(minor_tables, ignore_index=True) if minor_tables else pd.DataFrame()
    df_pair = pd.concat(pair_tables, ignore_index=True) if pair_tables else pd.DataFrame()

    # 导出 CSV
    df_main.to_csv(OUT_DIR / "main_metal_element_summary_by_model.csv", index=False, encoding="utf-8-sig")
    df_minor.to_csv(OUT_DIR / "minor_metal_element_summary_by_model.csv", index=False, encoding="utf-8-sig")
    df_pair.to_csv(OUT_DIR / "metal_pair_summary_by_model.csv", index=False, encoding="utf-8-sig")

    # 导出 Excel
    xlsx_path = OUT_DIR / "all_element_level_summary.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df_main.to_excel(writer, sheet_name="main_metal_by_model", index=False)
        df_minor.to_excel(writer, sheet_name="minor_metal_by_model", index=False)
        df_pair.to_excel(writer, sheet_name="metal_pair_by_model", index=False)

    # 说明
    notes = {
        "meaning": {
            "mean_shap": "平均SHAP，>0 表示该元素整体更推向好活性",
            "mean_abs_shap": "平均绝对SHAP，表示该元素的重要性强弱",
            "pos_shap_rate": "该元素对应样本中，SHAP>0 的比例",
            "mean_p_good": "该元素对应样本的平均好活性预测概率",
            "pos_class_rate": "该元素对应样本在训练标签中为好活性的比例"
        },
        "tips": [
            "优先看 mean_shap 和 n，同时参考 mean_abs_shap。",
            "如果某元素 n 很小（例如 1~2），只能当提示，不能下定论。",
            "如果 main 和 minor 同时都偏正，再结合 metal_pair_summary 看组合。"
        ]
    }
    with open(OUT_DIR / "README_element_summary.json", "w", encoding="utf-8") as f:
        json.dump(notes, f, ensure_ascii=False, indent=2)

    print("=" * 80)
    print("Done.")
    print("Outputs:")
    print(OUT_DIR / "main_metal_element_summary_by_model.csv")
    print(OUT_DIR / "minor_metal_element_summary_by_model.csv")
    print(OUT_DIR / "metal_pair_summary_by_model.csv")
    print(OUT_DIR / "all_element_level_summary.xlsx")
    print("=" * 80)


if __name__ == "__main__":
    main()

[READ] CAT_KcatHigh: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\CAT_KcatHigh\sample_level_shap.csv
[READ] CAT_KmLow: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\CAT_KmLow\sample_level_shap.csv
[READ] CAT_VmaxHigh: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\CAT_VmaxHigh\sample_level_shap.csv
[READ] SOD_IC50Low: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\SOD_IC50Low\sample_level_shap.csv
[READ] SOD_KmLow: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\SOD_KmLow\sample_level_shap.csv
[READ] SOD_VmaxHigh: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\SOD_VmaxHigh\sample_level_shap.csv
Done.
Outputs:
C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\element_level_summary\main_metal_element_summary_by_model.csv
C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\element_level_summary\minor_metal_element_summary_by_model.csv
C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\element_level_summary\metal_pair_s

In [16]:
# -*- coding: utf-8 -*-


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# 0) 路径
# =========================================================
BASE_DIR = Path(r"C:\Users\86158\Desktop\all6_classifier_raw_shap_summary")
ELEMENT_DIR = BASE_DIR / "element_level_summary"
OUT_DIR = BASE_DIR / "summary_plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAIN_METAL_CSV = ELEMENT_DIR / "main_metal_element_summary_by_model.csv"
MINOR_METAL_CSV = ELEMENT_DIR / "minor_metal_element_summary_by_model.csv"
PAIR_CSV = ELEMENT_DIR / "metal_pair_summary_by_model.csv"

NUMERIC_BINS_CSV = BASE_DIR / "combined_numeric_feature_bins.csv"
CATEGORICAL_CSV = BASE_DIR / "combined_categorical_feature_levels.csv"


# =========================================================
# 1) 安全转换函数
# =========================================================
def safe_str_series(s: pd.Series) -> pd.Series:
    """把任意列安全转成字符串，避免 matplotlib 分类轴报错"""
    return s.astype("object").where(~pd.isna(s), "NA").astype(str)


def safe_float_series(s: pd.Series) -> pd.Series:
    """把任意列安全转成 float"""
    return pd.to_numeric(s, errors="coerce").astype(float)


# =========================================================
# 2) 读数据
# =========================================================
def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"[WARN] File not found: {path}")
        return pd.DataFrame()
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except Exception as e:
        print(f"[WARN] Failed to read {path}: {e}")
        return pd.DataFrame()


df_main = safe_read_csv(MAIN_METAL_CSV)
df_minor = safe_read_csv(MINOR_METAL_CSV)
df_pair = safe_read_csv(PAIR_CSV)
df_num_bins = safe_read_csv(NUMERIC_BINS_CSV)
df_cat = safe_read_csv(CATEGORICAL_CSV)


# =========================================================
# 3) 通用画图函数
# =========================================================
def save_hbar(df, label_col, value_col, title, out_png, topn=10):
    if df is None or df.empty:
        return

    d = df.copy()

    # 强制类型
    if label_col not in d.columns or value_col not in d.columns:
        return

    d[label_col] = safe_str_series(d[label_col])
    d[value_col] = safe_float_series(d[value_col])

    # 过滤空值
    d = d[d[label_col].notna() & d[value_col].notna()].copy()
    if d.empty:
        return

    d = d.sort_values(value_col, ascending=False).head(topn).copy()
    d = d.iloc[::-1]

    labels = d[label_col].astype(str).tolist()
    values = d[value_col].astype(float).tolist()

    if len(labels) == 0:
        return

    plt.figure(figsize=(8, max(4, 0.45 * len(d))))
    plt.barh(labels, values)
    plt.xlabel(str(value_col))
    plt.title(str(title))
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


def save_lineplot(df, x_col, y_col, group_col, title, out_png):
    if df is None or df.empty:
        return

    d = df.copy()
    for c in [x_col, y_col, group_col]:
        if c not in d.columns:
            return

    d[x_col] = safe_float_series(d[x_col])
    d[y_col] = safe_float_series(d[y_col])
    d[group_col] = safe_str_series(d[group_col])

    d = d[d[x_col].notna() & d[y_col].notna() & d[group_col].notna()].copy()
    if d.empty:
        return

    models = d[group_col].dropna().unique().tolist()

    plt.figure(figsize=(8, 5))
    for m in models:
        sub = d[d[group_col] == m].copy().sort_values(x_col)
        if sub.empty:
            continue
        plt.plot(
            sub[x_col].astype(float).values,
            sub[y_col].astype(float).values,
            marker="o",
            label=str(m)
        )

    plt.axhline(0, linewidth=0.8)
    plt.xlabel(str(x_col))
    plt.ylabel(str(y_col))
    plt.title(str(title))
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


def save_heatmap(pivot_df, title, out_png):
    if pivot_df is None or pivot_df.empty:
        return

    d = pivot_df.copy()

    # 行列标签统一转字符串
    d.index = pd.Index([str(x) for x in d.index])
    d.columns = pd.Index([str(x) for x in d.columns])

    # 数值矩阵强制 float
    try:
        arr = d.apply(pd.to_numeric, errors="coerce").values.astype(float)
    except Exception:
        return

    if arr.size == 0:
        return

    plt.figure(figsize=(max(6, 0.6 * d.shape[1]), max(4, 0.45 * d.shape[0])))
    im = plt.imshow(arr, aspect="auto")
    plt.colorbar(im, label="Weighted mean SHAP")
    plt.xticks(range(d.shape[1]), d.columns.tolist(), rotation=45, ha="right")
    plt.yticks(range(d.shape[0]), d.index.tolist())
    plt.title(str(title))
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


# =========================================================
# 4) 主金属 / 辅金属 元素排行图
# =========================================================
if not df_main.empty and "model_name" in df_main.columns:
    for model_name in sorted(df_main["model_name"].dropna().astype(str).unique()):
        sub = df_main[(df_main["model_name"].astype(str) == model_name) & (safe_float_series(df_main["n"]) >= 1)].copy()
        if not sub.empty:
            sub["label"] = safe_str_series(sub["element_symbol"]) + " (n=" + safe_str_series(sub["n"]) + ")"
            save_hbar(
                sub,
                label_col="label",
                value_col="mean_shap",
                title=f"{model_name} | Main metal positive elements",
                out_png=OUT_DIR / f"{model_name}_main_metal_top.png",
                topn=10
            )

if not df_minor.empty and "model_name" in df_minor.columns:
    for model_name in sorted(df_minor["model_name"].dropna().astype(str).unique()):
        sub = df_minor[(df_minor["model_name"].astype(str) == model_name) & (safe_float_series(df_minor["n"]) >= 1)].copy()
        if not sub.empty:
            sub["label"] = safe_str_series(sub["element_symbol"]) + " (n=" + safe_str_series(sub["n"]) + ")"
            save_hbar(
                sub,
                label_col="label",
                value_col="mean_shap",
                title=f"{model_name} | Minor metal positive elements",
                out_png=OUT_DIR / f"{model_name}_minor_metal_top.png",
                topn=10
            )


# =========================================================
# 5) 主-辅金属组合排行图
# =========================================================
if not df_pair.empty and "model_name" in df_pair.columns:
    for model_name in sorted(df_pair["model_name"].dropna().astype(str).unique()):
        sub = df_pair[(df_pair["model_name"].astype(str) == model_name) & (safe_float_series(df_pair["n"]) >= 1)].copy()
        if not sub.empty:
            sub["pair_label"] = (
                safe_str_series(sub["main_element_symbol"]) + "–" +
                safe_str_series(sub["minor_element_symbol"]) +
                " (n=" + safe_str_series(sub["n"]) + ")"
            )
            save_hbar(
                sub,
                label_col="pair_label",
                value_col="mean_pair_shap",
                title=f"{model_name} | Positive metal pairs",
                out_png=OUT_DIR / f"{model_name}_metal_pair_top.png",
                topn=10
            )


# =========================================================
# 6) 周期 / 族 热图
#    用 n 加权 mean_shap
# =========================================================
def weighted_mean(df, value_col="mean_shap", weight_col="n"):
    if df is None or df.empty:
        return np.nan
    try:
        w = safe_float_series(df[weight_col]).values
        v = safe_float_series(df[value_col]).values
    except Exception:
        return np.nan

    mask = np.isfinite(w) & np.isfinite(v)
    if mask.sum() == 0:
        return np.nan

    w = w[mask]
    v = v[mask]

    if np.sum(w) <= 0:
        return np.nan
    return np.sum(v * w) / np.sum(w)


# 主金属：period heatmap
period_rows = []
if not df_main.empty and all(c in df_main.columns for c in ["model_name", "period", "mean_shap", "n"]):
    for (model_name, period), sub in df_main.groupby(["model_name", "period"], dropna=False):
        if pd.isna(period):
            continue
        period_rows.append({
            "model_name": str(model_name),
            "period": int(period),
            "weighted_mean_shap": weighted_mean(sub)
        })

df_period = pd.DataFrame(period_rows)
if not df_period.empty:
    pivot_period = df_period.pivot(index="model_name", columns="period", values="weighted_mean_shap").sort_index()
    save_heatmap(pivot_period, "Main metal | Period heatmap", OUT_DIR / "main_metal_period_heatmap.png")


# 主金属：group heatmap
group_rows = []
if not df_main.empty and all(c in df_main.columns for c in ["model_name", "group", "mean_shap", "n"]):
    for (model_name, group), sub in df_main.groupby(["model_name", "group"], dropna=False):
        if pd.isna(group):
            continue
        group_rows.append({
            "model_name": str(model_name),
            "group": int(group),
            "weighted_mean_shap": weighted_mean(sub)
        })

df_group = pd.DataFrame(group_rows)
if not df_group.empty:
    pivot_group = df_group.pivot(index="model_name", columns="group", values="weighted_mean_shap").sort_index()
    save_heatmap(pivot_group, "Main metal | Group heatmap", OUT_DIR / "main_metal_group_heatmap.png")


# =========================================================
# 7) 数值特征区间趋势图
# =========================================================
focus_numeric = [
    "main metal number",
    "minor metal number",
    "size (nm)",
    "pH",
    "main metal proportion"
]

if not df_num_bins.empty and all(c in df_num_bins.columns for c in ["feature", "feature_mean", "shap_mean", "model_name"]):
    for feat in focus_numeric:
        sub = df_num_bins[df_num_bins["feature"].astype(str) == feat].copy()
        if not sub.empty:
            save_lineplot(
                sub,
                x_col="feature_mean",
                y_col="shap_mean",
                group_col="model_name",
                title=f"{feat} | SHAP trend by bin",
                out_png=OUT_DIR / f"numeric_trend_{feat.replace('/', '_')}.png"
            )


# =========================================================
# 8) 类别变量条形图
# =========================================================
focus_cat = ["shape", "surface treatment", "dispersion medium", "Substrate"]

if not df_cat.empty and all(c in df_cat.columns for c in ["model_name", "feature", "level", "n", "mean_shap"]):
    for model_name in sorted(df_cat["model_name"].dropna().astype(str).unique()):
        for feat in focus_cat:
            sub = df_cat[
                (df_cat["model_name"].astype(str) == model_name) &
                (df_cat["feature"].astype(str) == feat)
            ].copy()
            if not sub.empty:
                sub["label"] = safe_str_series(sub["level"]) + " (n=" + safe_str_series(sub["n"]) + ")"
                save_hbar(
                    sub,
                    label_col="label",
                    value_col="mean_shap",
                    title=f"{model_name} | {feat}",
                    out_png=OUT_DIR / f"{model_name}_{feat.replace('/', '_')}_levels.png",
                    topn=10
                )


print("=" * 80)
print("Done.")
print("Plots saved to:")
print(OUT_DIR)
print("=" * 80)

Done.
Plots saved to:
C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\summary_plots


In [18]:
# -*- coding: utf-8 -*-
"""
更直观的元素级热图（简洁版）：
- 横轴：具体元素（按原子序数）
- 纵轴：模型
- 颜色：mean_shap
- 低样本格子自动标灰
- 不显示格子内数值
- 如果某个元素在当前三行模型中全为空白，则不保留该元素列
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

BASE_DIR = Path(r"C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\element_level_summary")
OUT_DIR = BASE_DIR / "pretty_heatmaps_clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAIN_CSV = BASE_DIR / "main_metal_element_summary_by_model.csv"
MINOR_CSV = BASE_DIR / "minor_metal_element_summary_by_model.csv"

df_main = pd.read_csv(MAIN_CSV, encoding="utf-8-sig")
df_minor = pd.read_csv(MINOR_CSV, encoding="utf-8-sig")


def draw_element_heatmap(
    df,
    model_order,
    title,
    out_png,
    min_n_for_show=2,
    top_n_elements=12
):
    """
    df 需要包含：
    model_name, atomic_number, element_symbol, mean_shap, n
    """
    if df is None or df.empty:
        return

    d = df.copy()
    d["mean_shap"] = pd.to_numeric(d["mean_shap"], errors="coerce")
    d["n"] = pd.to_numeric(d["n"], errors="coerce")
    d["atomic_number"] = pd.to_numeric(d["atomic_number"], errors="coerce")
    d["model_name"] = d["model_name"].astype(str)
    d["element_symbol"] = d["element_symbol"].astype(str)

    d = d.dropna(subset=["model_name", "atomic_number", "element_symbol", "mean_shap", "n"]).copy()
    if d.empty:
        return

    # 只保留当前这些模型
    d = d[d["model_name"].isin(model_order)].copy()
    if d.empty:
        return

    # 先筛一轮全局“重要元素”
    d["abs_mean_shap"] = d["mean_shap"].abs()

    elem_score = (
        d.groupby(["atomic_number", "element_symbol"], as_index=False)
         .agg(score=("abs_mean_shap", "mean"), total_n=("n", "sum"))
         .sort_values(["score", "total_n"], ascending=[False, False])
         .head(top_n_elements)
         .sort_values("atomic_number")
    )

    keep_elements = elem_score["element_symbol"].tolist()
    d = d[d["element_symbol"].isin(keep_elements)].copy()
    if d.empty:
        return

    # pivot
    heat = d.pivot_table(
        index="model_name",
        columns="element_symbol",
        values="mean_shap",
        aggfunc="mean"
    ).reindex(index=model_order)

    nmat = d.pivot_table(
        index="model_name",
        columns="element_symbol",
        values="n",
        aggfunc="sum"
    ).reindex(index=model_order)

    # 低样本格子置空
    plot_vals = heat.copy()
    plot_vals[(nmat < min_n_for_show)] = np.nan

    # 删除“在当前三行里全为空白”的元素列
    valid_cols = plot_vals.columns[plot_vals.notna().any(axis=0)].tolist()
    plot_vals = plot_vals[valid_cols].copy()
    nmat = nmat[valid_cols].copy()

    if plot_vals.empty or plot_vals.shape[1] == 0:
        return

    # 为了按原子序数排序，重新取元素顺序
    col_order_df = (
        d[["atomic_number", "element_symbol"]]
        .drop_duplicates()
        .sort_values("atomic_number")
    )
    col_order = [e for e in col_order_df["element_symbol"].tolist() if e in plot_vals.columns]

    plot_vals = plot_vals[col_order]
    nmat = nmat[col_order]

    vals = plot_vals.values.astype(float)
    vmax = np.nanmax(np.abs(vals))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 0.1
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    fig_w = max(7, 0.85 * len(col_order))
    fig_h = max(3.8, 0.9 * len(model_order))

    plt.figure(figsize=(fig_w, fig_h))
    cmap = plt.cm.RdBu_r.copy()
    cmap.set_bad(color="#DDDDDD")  # 缺失值灰色

    im = plt.imshow(plot_vals.values, aspect="auto", cmap=cmap, norm=norm)

    plt.xticks(range(len(col_order)), col_order, rotation=45, ha="right", fontsize=12)
    plt.yticks(range(len(model_order)), model_order, fontsize=13)
    plt.title(title, fontsize=22, pad=14)

    cbar = plt.colorbar(im)
    cbar.set_label("Mean SHAP", fontsize=15)

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()


# SOD / CAT 分开画
sod_models = ["SOD_IC50Low", "SOD_KmLow", "SOD_VmaxHigh"]
cat_models = ["CAT_KmLow", "CAT_VmaxHigh", "CAT_KcatHigh"]

# 主金属
draw_element_heatmap(
    df_main[df_main["model_name"].isin(sod_models)],
    model_order=sod_models,
    title="SOD | Main-metal element heatmap",
    out_png=OUT_DIR / "SOD_main_metal_element_heatmap.png",
    min_n_for_show=2,
    top_n_elements=12
)

draw_element_heatmap(
    df_main[df_main["model_name"].isin(cat_models)],
    model_order=cat_models,
    title="CAT | Main-metal element heatmap",
    out_png=OUT_DIR / "CAT_main_metal_element_heatmap.png",
    min_n_for_show=2,
    top_n_elements=12
)

# 辅金属
draw_element_heatmap(
    df_minor[df_minor["model_name"].isin(sod_models)],
    model_order=sod_models,
    title="SOD | Minor-metal element heatmap",
    out_png=OUT_DIR / "SOD_minor_metal_element_heatmap.png",
    min_n_for_show=2,
    top_n_elements=12
)

draw_element_heatmap(
    df_minor[df_minor["model_name"].isin(cat_models)],
    model_order=cat_models,
    title="CAT | Minor-metal element heatmap",
    out_png=OUT_DIR / "CAT_minor_metal_element_heatmap.png",
    min_n_for_show=2,
    top_n_elements=12
)

print("Done:", OUT_DIR)

Done: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\element_level_summary\pretty_heatmaps_clean


In [21]:
# -*- coding: utf-8 -*-
"""
单独绘制 SOD / CAT 的 size 与 pH 趋势图
--------------------------------------
输入：
C:\\Users\\86158\\Desktop\\all6_classifier_raw_shap_summary\\combined_numeric_feature_bins.csv

输出：
C:\\Users\\86158\\Desktop\\all6_classifier_raw_shap_summary\\trend_plots_only\\
    - SOD_size_trend.png
    - CAT_size_trend.png
    - SOD_pH_trend.png
    - CAT_pH_trend.png
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# 0) 路径
# =========================================================
BASE_DIR = Path(r"C:\Users\86158\Desktop\all6_classifier_raw_shap_summary")
INPUT_CSV = BASE_DIR / "combined_numeric_feature_bins.csv"
OUT_DIR = BASE_DIR / "trend_plots_only"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# 1) 读数据
# =========================================================
df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")


# =========================================================
# 2) 工具函数
# =========================================================
def save_family_trend_plot(
    df,
    feature_name,
    model_order,
    title,
    out_png,
    x_log=False
):
    """
    只画某个 feature 在指定模型集合中的趋势图
    - 横轴: feature_mean
    - 纵轴: shap_mean
    - 每张图只画 3 条线
    """
    if df is None or df.empty:
        return

    need_cols = ["feature", "feature_mean", "shap_mean", "model_name"]
    for c in need_cols:
        if c not in df.columns:
            print(f"[WARN] missing column: {c}")
            return

    d = df.copy()
    d["feature"] = d["feature"].astype(str)
    d["model_name"] = d["model_name"].astype(str)
    d["feature_mean"] = pd.to_numeric(d["feature_mean"], errors="coerce")
    d["shap_mean"] = pd.to_numeric(d["shap_mean"], errors="coerce")

    d = d[
        (d["feature"] == feature_name) &
        (d["model_name"].isin(model_order)) &
        d["feature_mean"].notna() &
        d["shap_mean"].notna()
    ].copy()

    if d.empty:
        print(f"[WARN] no data for {title}")
        return

    plt.figure(figsize=(7.2, 4.8))

    for model_name in model_order:
        sub = d[d["model_name"] == model_name].copy().sort_values("feature_mean")
        if sub.empty:
            continue

        x = sub["feature_mean"].astype(float).values
        y = sub["shap_mean"].astype(float).values

        if x_log:
            mask = np.isfinite(x) & np.isfinite(y) & (x > 0)
        else:
            mask = np.isfinite(x) & np.isfinite(y)

        x = x[mask]
        y = y[mask]

        if len(x) == 0:
            continue

        plt.plot(x, y, marker="o", linewidth=2, label=model_name)

    plt.axhline(0, linewidth=0.8)

    if x_log:
        plt.xscale("log")

    plt.xlabel(feature_name)
    plt.ylabel("mean SHAP")
    plt.title(title)
    plt.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"[OK] saved: {out_png}")


# =========================================================
# 3) 模型分组
# =========================================================
sod_models = ["SOD_IC50Low", "SOD_KmLow", "SOD_VmaxHigh"]
cat_models = ["CAT_KmLow", "CAT_VmaxHigh", "CAT_KcatHigh"]


# =========================================================
# 4) 画图
# =========================================================
save_family_trend_plot(
    df=df,
    feature_name="size (nm)",
    model_order=sod_models,
    title="SOD | size trend",
    out_png=OUT_DIR / "SOD_size_trend.png",
    x_log=True
)

save_family_trend_plot(
    df=df,
    feature_name="size (nm)",
    model_order=cat_models,
    title="CAT | size trend",
    out_png=OUT_DIR / "CAT_size_trend.png",
    x_log=True
)

save_family_trend_plot(
    df=df,
    feature_name="pH",
    model_order=sod_models,
    title="SOD | pH trend",
    out_png=OUT_DIR / "SOD_pH_trend.png",
    x_log=False
)

save_family_trend_plot(
    df=df,
    feature_name="pH",
    model_order=cat_models,
    title="CAT | pH trend",
    out_png=OUT_DIR / "CAT_pH_trend.png",
    x_log=False
)

print("=" * 72)
print("Done.")
print("Output dir:", OUT_DIR)
print("=" * 72)

[OK] saved: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\trend_plots_only\SOD_size_trend.png
[OK] saved: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\trend_plots_only\CAT_size_trend.png
[OK] saved: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\trend_plots_only\SOD_pH_trend.png
[OK] saved: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\trend_plots_only\CAT_pH_trend.png
Done.
Output dir: C:\Users\86158\Desktop\all6_classifier_raw_shap_summary\trend_plots_only
